***
##### 2.2 Diagnóstico do Modelo Inicial e Linha de Base (Etapas 1 a 3)

**Decisões metodológicas documentadas (grill-me e `PRE_ESPECIFICACAO.md`):**
1. **Amostra observada (`obs = TRUE`):** o dataframe completo (`df`, 3.384 linhas) contém 612 observações (18,1%) cujo desfecho `y1a_taxa_conversao__*` foi preenchido na base precária (50% interpolação linear municipal + 50% mediana da microrregião×ano). Nos modelos a seguir, restringimos a estimação principal às **2.772 observações efetivamente observadas (137 municípios)**.
2. **Tratamento de `multas_ibama`:** na base precária, 48% das linhas de `x3_n_autos_flora__ibama_espacial` receberam zero administrativo. Conforme o texto do artigo (Seção 2.3), ausências não equivalem a zero real. Por isso, retiramos os autos do modelo principal e os avaliamos: (i) na amostra fiscal elegível (`fisc = TRUE`, 1.140 obs, Seção 2.4) e (ii) por meio dos **embargos do IBAMA geocodificados** na amostra completa (Caminho B2, Seção 2.7).
3. **Padronização e escala (`ESCOLHENDO VARIAVEIS H1 PREC.txt`):**
   - Desfecho em pontos percentuais: `y_agr = 100 * y1a_taxa_conversao__agricultura` e `y_pas = 100 * y1a_taxa_conversao__pastagem`.
   - Preços reais em log natural centrados na média dos 24 anos (`lps`, `lpm`).
   - Aptidão GAEZ padronizada pelo desvio-padrão populacional dos 141 municípios (`zs`, `zm`).
   - Interações principais: `xs = lps * zs` e `xm = lpm * zm`.
   - Controle climático: `rain = x1c_v2__precip_anomaly_1981_2000 / 100` (em 100 mm).

In [1]:
# Pacotes adicionais para inferência HAC, leitura rápida de dados de 2_INTI/xdata/ e geoprocessamento (Caminhos A e B)
suppressPackageStartupMessages({
  library(tidyverse)
  library(fixest)
  library(sandwich)
  library(lmtest)
  library(data.table)
  library(sf)
})
options(width = 150)
setFixest_nthreads(1)
setFixest_notes(FALSE) # Oculta avisos repetitivos de singletons; o N exato consta em cada tabela
sf_use_s2(TRUE)

# Garantir diretório de trabalho e pasta de auditoria em 2_INTI/audit/
dir_modelos <- "C:/Users/ferna/Documents/dev/Tese USP/tese_eco_usp/2_INTI/chapters/chap1/stats/2. MODELOS/grupos_h1/2_base_completa_h1_precaria/MODELOS_H1_PREC"
if (!file.exists("base_modelos_h1_prec.csv")) setwd(dir_modelos)
root <- normalizePath(file.path(getwd(), "../../../../../../.."), winslash = "/")
stopifnot(basename(root) == "2_INTI")
out_dir <- file.path(root, "audit", "readequacao_artigo1_20261112", "saidas", "modelos_prec_v2")
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

# Funções auxiliares de transformação, estimação e formatação de tabelas
tf <- function(x) tolower(as.character(x)) == "true"
sd_pop <- function(x) sqrt(mean((x - mean(x, na.rm = TRUE))^2, na.rm = TRUE))

raw_v2 <- as_tibble(read.csv("base_modelos_h1_prec.csv", stringsAsFactors = FALSE))
stopifnot(nrow(raw_v2) == 3384, !anyDuplicated(raw_v2[c("z0_codigo_ibge", "z0_ano")]))

df_v2 <- raw_v2 %>% transmute(
  ibge = factor(z0_codigo_ibge),
  ibge_int = as.integer(z0_codigo_ibge),
  ano = as.integer(z0_ano),
  micro = factor(z0_microrregiao),
  bioma = factor(z0_bioma_predominante),
  amz = z0_biome_amazonia_share_i,
  cer = z0_biome_cerrado_share_i,
  pan = z0_biome_pantanal_share_i,
  taxa_agr = y1a_taxa_conversao__agricultura,
  y_agr = 100 * y1a_taxa_conversao__agricultura,
  y_pas = 100 * y1a_taxa_conversao__pastagem,
  area_soja_ha = x2b_area_plantada_ha__soja,
  area_milho_ha = x2b_area_plantada_ha__milho,
  ps_l1 = x2a_preco_real_usd2024_media_anual_lag1__soja,
  pm_l1 = x2a_preco_real_usd2024_media_anual_lag1__milho,
  ps_t = x2a_preco_real_usd2024_media_anual__soja,
  pm_t = x2a_preco_real_usd2024_media_anual__milho,
  dap_l1 = x2a_preco_real_usd2024_media_anual_lag1__dap,
  apt_s = x2b_capacidade_valor__soja__gaez_yxHr0_soy,
  apt_m = x2b_capacidade_valor__milho__gaez_yxHr0_mze,
  cst_s = x2d_custo_conab_soja_ogm_safra1_conab_h_usd2024,
  cst_m = x2d_custo_conab_milho_2a_ogm_safra0_conab_h_usd2024,
  rain = x1c_v2__precip_anomaly_1981_2000 / 100,
  autos_l1 = x3_n_autos_flora_elegiveis_lag1__ibama_espacial,
  obs = tf(x4_v2__flag_amostra_conversao_ambos),
  fisc = tf(x4_v2__flag_amostra_fiscal_lag1)
)

# Séries anuais (24 anos) e cadastro municipal (141 municípios) para centragem e padronização exatas
anos_v2 <- df_v2 %>% distinct(ano, ps_l1, pm_l1, ps_t, pm_t, dap_l1) %>% arrange(ano) %>%
  mutate(ps_lead = lead(ps_t), pm_lead = lead(pm_t)) # Preço em t+1 para teste de placebo
mun_v2 <- df_v2 %>% distinct(ibge, ibge_int, apt_s, apt_m)
stopifnot(nrow(anos_v2) == 24, nrow(mun_v2) == 141)

c_ps <- mean(log(anos_v2$ps_l1)); c_pm <- mean(log(anos_v2$pm_l1)); c_dap <- mean(log(anos_v2$dap_l1))

df_v2 <- df_v2 %>%
  left_join(anos_v2 %>% select(ano, ps_lead, pm_lead), by = "ano") %>%
  arrange(ibge, ano) %>%
  group_by(ibge) %>%
  mutate(cst_s_l1 = lag(cst_s), cst_m_l1 = lag(cst_m)) %>%
  ungroup() %>%
  mutate(
    lps = log(ps_l1) - c_ps,
    lpm = log(pm_l1) - c_pm,
    zs = (apt_s - mean(mun_v2$apt_s)) / sd_pop(mun_v2$apt_s),
    zm = (apt_m - mean(mun_v2$apt_m)) / sd_pop(mun_v2$apt_m),
    xs = lps * zs,
    xm = lpm * zm,
    trend = (ano - 2001) / 10,
    zs_tr = zs * trend,
    zm_tr = zm * trend,
    xs_f = (log(ps_lead) - c_ps) * zs,
    xm_f = (log(pm_lead) - c_pm) * zm,
    xs_amz = xs * amz, xs_pan = xs * pan,
    xm_amz = xm * amz, xm_pan = xm * pan,
    x_dap_s = (log(dap_l1) - c_dap) * zs,
    x_dap_m = (log(dap_l1) - c_dap) * zm,
    x_cst_s = (log(cst_s_l1) - mean(log(cst_s_l1), na.rm = TRUE)) * zs,
    x_cst_m = (log(cst_m_l1) - mean(log(cst_m_l1), na.rm = TRUE)) * zm,
    f1 = log1p(autos_l1),
    f1_ps = lps * f1, f1_zs = zs * f1, f1_xs = xs * f1,
    f1_pm = lpm * f1, f1_zm = zm * f1, f1_xm = xm * f1,
    post05 = as.numeric(ano >= 2005),
    post06 = as.numeric(ano >= 2006)
  )

d_obs  <- df_v2 %>% filter(obs)
d_fisc <- df_v2 %>% filter(fisc)
stopifnot(nrow(d_obs) == 2772, n_distinct(d_obs$ibge) == 137,
          nrow(d_fisc) == 1140, n_distinct(d_fisc$ibge) == 128)

bioma_amz <- levels(df_v2$bioma)[grepl("^Amaz", levels(df_v2$bioma))]
bioma_cer <- levels(df_v2$bioma)[grepl("^Cerr", levels(df_v2$bioma))]
bioma_pan <- levels(df_v2$bioma)[grepl("^Pant", levels(df_v2$bioma))]

# Lista global para registrar todos os coeficientes estimados e exportar CSV de auditoria
RES <- list()
rec <- function(m, id, grupo, descr, amostra = NA_character_) {
  ct <- as.data.frame(coeftable(m))
  r2_t <- tryCatch(as.numeric(r2(m, "r2")), error = function(e) NA_real_)
  r2_w <- tryCatch(as.numeric(r2(m, "wr2")), error = function(e) NA_real_)
  RES[[length(RES) + 1]] <<- tibble(
    id = id, grupo = grupo, descricao = descr, amostra = amostra,
    termo = rownames(ct), estimativa = ct[[1]], erro_padrao = ct[[2]], p = ct[[4]], n = nobs(m),
    r2_total = r2_t, r2_within = r2_w
  )
  invisible(m)
}

fit_run <- function(crop, data, y = "y_agr", fe = "ibge + bioma^ano", vcov = ~ibge,
                    trend = TRUE, rain = TRUE, extra = NULL) {
  x   <- if (crop == "soja") "xs" else "xm"
  ztr <- if (crop == "soja") "zs_tr" else "zm_tr"
  rhs <- c(x, if (trend) ztr, if (rain) "rain", extra)
  f <- as.formula(paste(y, "~", paste(rhs, collapse = " + "), "|", fe))
  feols(f, data = data, vcov = vcov)
}
run <- function(crop, data, id, grupo, descr, amostra, ...) {
  rec(fit_run(crop, data, ...), id, grupo, descr, amostra)
}
tab <- function(..., headers = NULL) {
  ms <- list(...)
  if (is.null(headers)) headers <- paste0("(", seq_along(ms), ")")
  estrelas <- function(p) ifelse(p < 0.01, "***", ifelse(p < 0.05, "**", ifelse(p < 0.10, "*", "")))
  termos <- unique(unlist(lapply(ms, function(m) rownames(coeftable(m)))))
  col <- function(m) {
    ct <- coeftable(m)
    vapply(termos, function(t) {
      if (t %in% rownames(ct)) {
        sprintf("%s%s (%s)", formatC(ct[t, 1], digits = 3, format = "g", flag = "#"),
                estrelas(ct[t, 4]), formatC(ct[t, 2], digits = 3, format = "g", flag = "#"))
      } else ""
    }, character(1))
  }
  out <- as.data.frame(do.call(cbind, lapply(ms, col)), stringsAsFactors = FALSE)
  colnames(out) <- headers; rownames(out) <- termos
  info <- do.call(cbind, lapply(ms, function(m) c(
    `N` = format(nobs(m), big.mark = ""),
    `R2 total` = sprintf("%.4f", as.numeric(r2(m, "r2"))),
    `R2 within` = sprintf("%.4f", as.numeric(r2(m, "wr2"))),
    `Efeitos fixos` = paste(m$fixef_vars, collapse = " + "))))
  colnames(info) <- headers
  print(rbind(out, as.data.frame(info, stringsAsFactors = FALSE)))
  cat("Estimativa (erro-padrão agrupado por município, salvo indicação); * p<0,10  ** p<0,05  *** p<0,01\n")
}
cat("Ambiente v2 configurado. Amostra observada:", nrow(d_obs), "obs (", n_distinct(d_obs$ibge), "municípios).\n")

Ambiente v2 configurado. Amostra observada: 2772 obs ( 137 municípios).


###### Transição passo a passo do `model_1a` para a especificação padronizada
As tabelas abaixo mostram o efeito isolado de cada correção sobre o `model_1a` (soja e milho):
- **(1) original:** replica exatamente o `model_1a` da seção 2.1 (3.384 obs, níveis, Y preenchida);
- **(2) Y observada:** filtra `obs = TRUE` (2.769 obs ativas; `risco_s` deixa de ser significativo, $p = 0,121$);
- **(3) sem autos:** retira `autos_l1` e `risco` (48% de zeros administrativos);
- **(4) escala notas:** passa para $Y$ em pontos percentuais e $\ln P_{t-1}$ centrado $\times$ aptidão padronizada ($Z$);
- **(5) + rain:** inclui anomalia de precipitação (`rain`).

In [2]:
nb_base <- df_v2 %>% mutate(
  taxa = taxa_agr,
  pot_s = ps_l1 * apt_s, pot_m = pm_l1 * apt_m,
  risco_s = ps_l1 * autos_l1, risco_m = pm_l1 * autos_l1
)

baseline_step <- function(crop) {
  p <- if (crop == "soja") list(pot = "pot_s", risco = "risco_s", x = "xs") else list(pot = "pot_m", risco = "risco_m", x = "xm")
  fm <- function(lhs, rhs) as.formula(paste(lhs, "~", rhs, "| ibge + ano"))
  m <- list(
    "1 original"      = feols(fm("taxa", paste(p$pot, "+ autos_l1 +", p$risco)), nb_base, vcov = ~ibge),
    "2 Y observada"   = feols(fm("taxa", paste(p$pot, "+ autos_l1 +", p$risco)), filter(nb_base, obs), vcov = ~ibge),
    "3 sem autos"     = feols(fm("taxa", p$pot), filter(nb_base, obs), vcov = ~ibge),
    "4 escala notas"  = feols(fm("y_agr", p$x), filter(nb_base, obs), vcov = ~ibge),
    "5 + rain"        = feols(fm("y_agr", paste(p$x, "+ rain")), filter(nb_base, obs), vcov = ~ibge)
  )
  for (i in seq_along(m)) {
    rec(m[[i]], paste0("B", i, "_", crop), "baseline", paste(crop, names(m)[i]), if (i == 1) "todas" else "obs")
  }
  cat("=== Linha de base:", toupper(crop), "===\n")
  tab(m[[1]], m[[2]], m[[3]], m[[4]], m[[5]], headers = names(m))
}
baseline_step("soja")
baseline_step("milho")

=== Linha de base: SOJA ===
                          1 original          2 Y observada           3 sem autos   4 escala notas            5 + rain
pot_s         6.07e-09*** (2.13e-09) 6.58e-09*** (2.51e-09) 6.08e-09** (2.39e-09)                                     
autos_l1       -2.21e-05* (1.30e-05)   -2.34e-05 (1.71e-05)                                                           
risco_s        4.81e-08** (2.39e-08)    5.14e-08 (3.29e-08)                                                           
xs                                                                                0.140** (0.0545)    0.136** (0.0532)
rain                                                                                               -0.0139** (0.00671)
N                               3384                   2769                  2769             2769                2769
R2 total                      0.3534                 0.3613                0.3607           0.3621              0.3650
R2 within           

=== Linha de base: MILHO ===
                          1 original          2 Y observada            3 sem autos     4 escala notas            5 + rain
pot_m         3.48e-09*** (9.40e-10) 4.15e-09*** (1.28e-09) 4.07e-09*** (1.26e-09)                                       
autos_l1        -4.42e-06 (8.40e-06)   -1.17e-06 (1.02e-05)                                                              
risco_m          3.31e-08 (3.38e-08)    1.86e-08 (4.26e-08)                                                              
xm                                                                                 0.0930*** (0.0280)  0.0903*** (0.0275)
rain                                                                                                  -0.0144** (0.00691)
N                               3384                   2769                   2769               2769                2769
R2 total                      0.3494                 0.3567                 0.3566             0.3574              0.

***
##### 2.3 Modelos Principais (M1 e M2), Moderação por Bioma e Colinearidade (M3) (Etapas 4, 5 e 6)

**Decisões implementadas:**
- **M1 (soja) e M2 (milho) separados como especificações principais:** incluem controle de tendência diferencial por aptidão (`zs_tr` / `zm_tr`), anomalia de precipitação (`rain`) e efeitos fixos de município e de **bioma predominante $\times$ ano** (`ibge + bioma^ano`), com erros-padrão agrupados por município.
- **Duplo papel do bioma:** a coluna categórica (`0_bioma_predominante`) absorve choques anuais específicos de cada bioma (`bioma^ano`) e define subamostras (Amazônia vs. Cerrado), enquanto as colunas contínuas de participação territorial (`0_biome_amazonia_share_i` e `0_biome_pantanal_share_i`, tendo o Cerrado como referência) entram nas interações de moderação (`M1b` e `M2b`), preservando os 51 municípios mistos.
- **M3 (soja e milho juntos) como robustez:** documenta a colinearidade entre os preços anuais ($r = 0,948$) e entre as aptidões municipais ($r = 0,591$).

In [3]:
# Etapa 4: Modelos principais M1 (soja) e M2 (milho)
M1 <- run("soja",  d_obs, "M1", "principal", "M1 soja principal", "obs")
M2 <- run("milho", d_obs, "M2", "principal", "M2 milho principal", "obs")
cat("=== MODELOS PRINCIPAIS (M1 e M2) ===\n")
tab(M1, M2, headers = c("M1 soja", "M2 milho"))

# Etapa 5: Moderação por bioma (shares contínuos dentro do modelo e subamostras por bioma predominante)
M1b <- run("soja",  d_obs, "M1b", "moderacao", "M1b soja x share bioma", "obs", extra = c("xs_amz", "xs_pan"))
M2b <- run("milho", d_obs, "M2b", "moderacao", "M2b milho x share bioma", "obs", extra = c("xm_amz", "xm_pan"))
cat("\n=== MODERAÇÃO CONTÍNUA POR SHARE DE BIOMA (Referência = Cerrado) ===\n")
tab(M1b, M2b, headers = c("M1b soja", "M2b milho"))

sub <- list()
for (crop in c("soja", "milho")) {
  for (b in c(bioma_amz, bioma_cer)) {
    d <- filter(d_obs, bioma == b)
    sub[[paste(crop, b)]] <- run(crop, d, paste0("S_", crop, "_", substr(b, 1, 3)), "subamostra",
                                 paste(crop, "subamostra", b), "obs", fe = "ibge + ano")
  }
}
cat("\n=== SUBAMOSTRAS POR BIOMA PREDOMINANTE (Amazônia vs. Cerrado) ===\n")
tab(sub[[1]], sub[[2]], sub[[3]], sub[[4]], headers = names(sub))

# Etapa 6: M3 (soja + milho juntos) e diagnóstico de colinearidade
M3 <- feols(y_agr ~ xs + xm + zs_tr + zm_tr + rain | ibge + bioma^ano, data = d_obs, vcov = ~ibge)
rec(M3, "M3", "robustez_cultura", "M3 soja + milho juntos", "obs")
cat("\n=== M3: SOJA E MILHO JUNTOS ===\n")
tab(M3, headers = "M3 soja + milho")

w3 <- wald(M3, keep = "^x[sm]$", print = FALSE)
rs <- resid(feols(xs ~ 1 | ibge + ano, d_obs)); rm <- resid(feols(xm ~ 1 | ibge + ano, d_obs))
cat(sprintf("Teste Wald conjunto (xs = xm = 0): F = %.3f (p = %.4f)\n", w3$stat, w3$p))
cat(sprintf("Correlações: preços lag1 (24 anos) = %.3f | aptidões (141 mun.) = %.3f | xs-xm within = %.3f\n",
            cor(anos_v2$ps_l1, anos_v2$pm_l1), cor(mun_v2$apt_s, mun_v2$apt_m), cor(rs, rm)))
RES[[length(RES) + 1]] <- tibble(id = "M3_wald", grupo = "robustez_cultura", descricao = "teste conjunto xs=xm=0",
  amostra = "obs", termo = "wald_xs_xm", estimativa = w3$stat, erro_padrao = NA_real_, p = w3$p, n = nobs(M3))

=== MODELOS PRINCIPAIS (M1 e M2) ===


                          M1 soja            M2 milho
xs              0.0848** (0.0358)                    
zs_tr          0.0405*** (0.0144)                    
rain          -0.0135** (0.00626) -0.0152** (0.00671)
xm                                 0.0519*** (0.0179)
zm_tr                              0.0538*** (0.0152)
N                            2767                2767
R2 total                   0.3828              0.3807
R2 within                  0.0198              0.0165
Efeitos fixos    ibge + bioma^ano    ibge + bioma^ano
Estimativa (erro-padrão agrupado por município, salvo indicação); * p<0,10  ** p<0,05  *** p<0,01



=== MODERAÇÃO CONTÍNUA POR SHARE DE BIOMA (Referência = Cerrado) ===


                         M1b soja           M2b milho
xs                0.0662 (0.0502)                    
zs_tr          0.0405*** (0.0144)                    
rain          -0.0134** (0.00625) -0.0152** (0.00669)
xs_amz             0.0461 (0.121)                    
xs_pan              0.175 (0.158)                    
xm                                  0.0617** (0.0255)
zm_tr                              0.0538*** (0.0151)
xm_amz                                -0.0127 (0.113)
xm_pan                             -0.120*** (0.0420)
N                            2767                2767
R2 total                   0.3829              0.3809
R2 within                  0.0200              0.0167
Efeitos fixos    ibge + bioma^ano    ibge + bioma^ano
Estimativa (erro-padrão agrupado por município, salvo indicação); * p<0,10  ** p<0,05  *** p<0,01



=== SUBAMOSTRAS POR BIOMA PREDOMINANTE (Amazônia vs. Cerrado) ===


                   soja Amazônia       soja Cerrado     milho Amazônia      milho Cerrado
xs               0.0866 (0.0610)   0.0752* (0.0439)                                      
zs_tr           0.0396* (0.0232)  0.0473** (0.0183)                                      
rain          -0.0230** (0.0104) -0.00232 (0.00579) -0.0241** (0.0108) -0.00484 (0.00728)
xm                                                     0.0185 (0.0514) 0.0572*** (0.0192)
zm_tr                                                  0.0426 (0.0361) 0.0623*** (0.0175)
N                           1303               1412               1303               1412
R2 total                  0.3735             0.3924             0.3700             0.3968
R2 within                 0.0149             0.0492             0.0094             0.0560
Efeitos fixos         ibge + ano         ibge + ano         ibge + ano         ibge + ano
Estimativa (erro-padrão agrupado por município, salvo indicação); * p<0,10  ** p<0,05  *** p<0,01



=== M3: SOJA E MILHO JUNTOS ===


                  M3 soja + milho
xs               0.0815* (0.0433)
xm               0.00601 (0.0239)
zs_tr             0.0247 (0.0198)
zm_tr             0.0346 (0.0220)
rain          -0.0137** (0.00623)
N                            2767
R2 total                   0.3841
R2 within                  0.0218
Efeitos fixos    ibge + bioma^ano
Estimativa (erro-padrão agrupado por município, salvo indicação); * p<0,10  ** p<0,05  *** p<0,01


Teste Wald conjunto (xs = xm = 0): F = 4.122 (p = 0.0183)


Correlações: preços lag1 (24 anos) = 0.948 | aptidões (141 mun.) = 0.591 | xs-xm within = 0.565


***
##### 2.4 Bateria de Robustez: Inferência, Placebo, Janelas, Desfecho, Custos e Amostra Fiscal (Etapas 7 a 11)

- **Inferência (Etapa 7):** cluster duplo (`ibge + ano`), cluster por microrregião (22 clusters), FE `ibge + micro^ano`, regressão em 24 pontos anuais (Newey-West HAC lag 2) e *leave-one-year-out* (excluindo 1 ano por vez).
- **Placebo e janelas (Etapa 8):** inclusão do preço futuro ($\ln P_{t+1} \times Z$) e recorte pós-ruptura (2006–2024).
- **Desfecho e amostra (Etapa 9):** winsorização a 99%, $\text{asinh}(10Y)$, exclusão do Pantanal, amostra preenchida (3.384 obs) e conversão para pastagem (`y_pas`) como destino comparativo.
- **Custos na base (Etapa 11) e Autos na amostra fiscal (Etapa 10):** avaliação exploratória de $\ln\text{DAP}_{t-1}\times Z$ e $\ln\text{CONAB}_{t-1}\times Z$, e estimação de M6/M7 nos 1.135 pares município-ano elegíveis da amostra fiscal.

In [4]:
# Etapa 7: Inferência alternativa
for (crop in c("soja", "milho")) {
  r1 <- run(crop, d_obs, paste0("R_cl2_", crop), "robustez_inferencia", paste(crop, "cluster municipio+ano"), "obs", vcov = ~ibge + ano)
  r2 <- run(crop, d_obs, paste0("R_clmic_", crop), "robustez_inferencia", paste(crop, "cluster microrregiao"), "obs", vcov = ~micro)
  r3 <- run(crop, d_obs, paste0("R_micano_", crop), "robustez_inferencia", paste(crop, "FE microrregiao x ano"), "obs", fe = "ibge + micro^ano")
  cat("=== Inferência alternativa:", toupper(crop), "===\n")
  tab(r1, r2, r3, headers = paste(crop, c("cluster mun+ano", "cluster micro", "FE micro x ano")))
}

# Regressão em 24 pontos anuais (Newey-West HAC lag 2) e Leave-One-Year-Out
two_step <- function(crop) {
  d <- d_obs %>% group_by(ibge) %>% mutate(ydm = y_agr - mean(y_agr)) %>% ungroup()
  d$zv  <- if (crop == "soja") d$zs else d$zm
  d$lpv <- if (crop == "soja") d$lps else d$lpm
  sl <- d %>% group_by(ano) %>% summarise(slope = cov(ydm, zv) / var(zv), lp = first(lpv), .groups = "drop")
  m <- lm(slope ~ lp, data = sl)
  ct <- coeftest(m, vcov. = NeweyWest(m, lag = 2, prewhite = FALSE))
  RES[[length(RES) + 1]] <<- tibble(id = paste0("R_2step_", crop), grupo = "robustez_inferencia",
    descricao = paste(crop, "regressao em 24 pontos (HAC lag 2)"), amostra = "obs",
    termo = "lp", estimativa = ct["lp", 1], erro_padrao = ct["lp", 2], p = ct["lp", 4], n = nrow(sl))
  cat(sprintf("Regressão 24 pontos (%s): coef = %.4f (EP HAC = %.4f, p = %.4f)\n", crop, ct["lp", 1], ct["lp", 2], ct["lp", 4]))
}
loyo <- function(crop) {
  x <- if (crop == "soja") "xs" else "xm"
  out <- map_dfr(sort(unique(d_obs$ano)), function(a) {
    m <- fit_run(crop, filter(d_obs, ano != a))
    rec(m, paste0("R_loyo_", crop, "_", a), "robustez_loyo", paste(crop, "exclui ano", a), "obs")
    ct <- coeftable(m)
    tibble(ano_excluido = a, estimativa = ct[x, 1], p = ct[x, 4])
  })
  cat(sprintf("Leave-one-year-out (%s): min = %.4f, max = %.4f | positivos = %d/24 | p < 0,10 = %d/24\n",
              crop, min(out$estimativa), max(out$estimativa), sum(out$estimativa > 0), sum(out$p < 0.10)))
}
two_step("soja"); two_step("milho")
loyo("soja"); loyo("milho")

=== Inferência alternativa: SOJA ===
              soja cluster mun+ano soja cluster micro soja FE micro x ano
xs                0.0848* (0.0481)    0.0848 (0.0567)     0.0631 (0.0456)
zs_tr            0.0405** (0.0165)    0.0405 (0.0238)     0.0253 (0.0196)
rain             -0.0135 (0.00866)  -0.0135 (0.00860)   -0.0111 (0.00939)
N                             2767               2767                2755
R2 total                    0.3828             0.3828              0.6154
R2 within                   0.0198             0.0198              0.0060
Efeitos fixos     ibge + bioma^ano   ibge + bioma^ano    ibge + micro^ano
Estimativa (erro-padrão agrupado por município, salvo indicação); * p<0,10  ** p<0,05  *** p<0,01
=== Inferência alternativa: MILHO ===
              milho cluster mun+ano milho cluster micro milho FE micro x ano
xm                 0.0519* (0.0277)    0.0519* (0.0296)     0.00406 (0.0194)
zm_tr            0.0538*** (0.0192)   0.0538** (0.0244)     0.0251* (0.0135)
rain

Regressão 24 pontos (soja): coef = 0.1344 (EP HAC = 0.0655, p = 0.0524)


Regressão 24 pontos (milho): coef = 0.0967 (EP HAC = 0.0506, p = 0.0691)


Leave-one-year-out (soja): min = 0.0490, max = 0.1011 | positivos = 24/24 | p < 0,10 = 24/24


Leave-one-year-out (milho): min = 0.0407, max = 0.0652 | positivos = 24/24 | p < 0,10 = 24/24


In [5]:
# Etapas 8 e 9: Placebo (preço t+1), janelas 2006-2024 e transformações de Y / amostra
q99 <- quantile(d_obs$y_agr, 0.99)
d_y <- d_obs %>% mutate(y_w = pmin(y_agr, q99), y_a = asinh(10 * y_agr))

for (crop in c("soja", "milho")) {
  xf <- if (crop == "soja") "xs_f" else "xm_f"
  p1 <- run(crop, d_obs, paste0("R_plac_", crop), "placebo", paste(crop, "placebo preco t+1"), "obs", extra = xf)
  w1 <- run(crop, filter(d_obs, ano >= 2006), paste0("R_w06_", crop), "janela", paste(crop, "2006-2024 com controle de ruptura"), "obs 2006-2024")
  w2 <- run(crop, filter(d_obs, ano >= 2006), paste0("R_w06nt_", crop), "janela", paste(crop, "2006-2024 sem controle de ruptura"), "obs 2006-2024", trend = FALSE)
  w3 <- run(crop, d_obs, paste0("R_nt_", crop), "janela", paste(crop, "2001-2024 sem controle de ruptura"), "obs", trend = FALSE)
  cat("=== Placebo e Janelas:", toupper(crop), "===\n")
  tab(p1, w1, w2, w3, headers = paste(crop, c("placebo t+1", "2006-24 c/ tend.", "2006-24 s/ tend.", "2001-24 s/ tend.")))

  a <- run(crop, d_y, paste0("R_win_", crop), "robustez_y", paste(crop, "Y winsor 99%"), "obs", y = "y_w")
  b <- run(crop, d_y, paste0("R_asinh_", crop), "robustez_y", paste(crop, "asinh(10Y)"), "obs", y = "y_a")
  c <- run(crop, filter(d_obs, bioma != bioma_pan), paste0("R_nopan_", crop), "robustez_amostra", paste(crop, "sem Pantanal"), "obs sem Pantanal")
  d <- run(crop, df_v2, paste0("R_full_", crop), "robustez_amostra", paste(crop, "Y preenchida (3.384)"), "todas")
  e <- run(crop, d_obs, paste0("R_pas_", crop), "pastagem", paste(crop, "Y = pastagem"), "obs", y = "y_pas")
  cat("=== Desfecho e Amostra:", toupper(crop), "===\n")
  tab(a, b, c, d, e, headers = paste(crop, c("winsor 99%", "asinh", "sem Pantanal", "Y preenchida", "Y pastagem")))
}

=== Placebo e Janelas: SOJA ===
                soja placebo t+1 soja 2006-24 c/ tend. soja 2006-24 s/ tend. soja 2001-24 s/ tend.
xs             0.0559** (0.0261)    -0.00671 (0.00828)    -0.00528 (0.00848)     0.142*** (0.0538)
zs_tr         0.0428*** (0.0153)    0.00993* (0.00535)                                            
rain          -0.0109* (0.00598)   -0.000881 (0.00154)    -0.00136 (0.00160)   -0.0159** (0.00669)
xs_f           0.0844** (0.0338)                                                                  
N                           2633                  2189                  2189                  2767
R2 total                  0.3883                0.5143                0.5131                0.3799
R2 within                 0.0206                0.0030                0.0006                0.0151
Efeitos fixos   ibge + bioma^ano      ibge + bioma^ano      ibge + bioma^ano      ibge + bioma^ano
Estimativa (erro-padrão agrupado por município, salvo indicação); * p<0,10  *

In [6]:
# Etapas 10 e 11: Custos presentes na base (DAP e CONAB) e Autos de Flora na amostra fiscal (N = 1.135)
for (crop in c("soja", "milho")) {
  dap <- if (crop == "soja") "x_dap_s" else "x_dap_m"
  cst <- if (crop == "soja") "x_cst_s" else "x_cst_m"
  a <- run(crop, d_obs, paste0("C_dap_", crop), "custos", paste(crop, "+ DAP x aptidao"), "obs", extra = dap)
  b <- run(crop, d_obs, paste0("C_cst_", crop), "custos", paste(crop, "+ custo CONAB x aptidao"), "obs", extra = cst)
  c <- run(crop, d_obs, paste0("C_both_", crop), "custos", paste(crop, "+ DAP e custo CONAB"), "obs", extra = c(dap, cst))
  cat("=== Custos da base (DAP e CONAB):", toupper(crop), "===\n")
  tab(a, b, c, headers = paste(crop, c("+ DAP", "+ CONAB", "+ ambos")))
}

F3 <- feols(y_agr ~ xs + xm + rain | ibge + ano, d_fisc, vcov = ~ibge)
rec(F3, "F_M3", "instituicoes", "M3 na amostra fiscal", "fiscal")
fit_f <- function(x, extra) feols(as.formula(paste("y_agr ~", paste(c(x, extra, "rain"), collapse = " + "), "| ibge + ano")), d_fisc, vcov = ~ibge)
F6s <- fit_f("xs", "f1");                            rec(F6s, "F_M6_soja",  "instituicoes", "M6 soja",  "fiscal")
F7s <- fit_f("xs", c("f1", "f1_ps", "f1_zs", "f1_xs")); rec(F7s, "F_M7_soja",  "instituicoes", "M7 soja",  "fiscal")
F6m <- fit_f("xm", "f1");                            rec(F6m, "F_M6_milho", "instituicoes", "M6 milho", "fiscal")
F7m <- fit_f("xm", c("f1", "f1_pm", "f1_zm", "f1_xm")); rec(F7m, "F_M7_milho", "instituicoes", "M7 milho", "fiscal")
cat("=== Autos de Flora na Amostra Fiscal Elegível (fisc = TRUE, N = 1.135) ===\n")
tab(F3, F6s, F7s, F6m, F7m, headers = c("M3 fiscal", "M6 soja", "M7 soja", "M6 milho", "M7 milho"))

=== Custos da base (DAP e CONAB): SOJA ===
                       soja + DAP         soja + CONAB         soja + ambos
xs              -0.00958 (0.0296)      0.0292 (0.0184)    -0.0655* (0.0353)
zs_tr          0.0368*** (0.0132)   0.0291*** (0.0108)  0.0254*** (0.00957)
rain          -0.0140** (0.00631) -0.00788** (0.00398) -0.00846** (0.00403)
x_dap_s         0.0688** (0.0264)                         0.0688** (0.0270)
x_cst_s                              0.120*** (0.0426)    0.122*** (0.0431)
N                            2767                 2650                 2650
R2 total                   0.3835               0.3769               0.3777
R2 within                  0.0209               0.0215               0.0228
Efeitos fixos    ibge + bioma^ano     ibge + bioma^ano     ibge + bioma^ano
Estimativa (erro-padrão agrupado por município, salvo indicação); * p<0,10  ** p<0,05  *** p<0,01
=== Custos da base (DAP e CONAB): MILHO ===
                      milho + DAP       milho + CONAB  

=== Autos de Flora na Amostra Fiscal Elegível (fisc = TRUE, N = 1.135) ===


                       M3 fiscal            M6 soja            M7 soja           M6 milho           M7 milho
xs               0.0414 (0.0514)    0.0494 (0.0448)    0.0935 (0.0694)                                      
xm               0.0118 (0.0176)                                         0.0323* (0.0195)    0.0384 (0.0265)
rain          -0.00609 (0.00398) -0.00615 (0.00391) -0.00628 (0.00401) -0.00567 (0.00371) -0.00575 (0.00374)
f1                               0.000192 (0.00597) -0.00254 (0.00560) 8.49e-05 (0.00597) -0.00188 (0.00599)
f1_ps                                                 0.0467* (0.0274)                                      
f1_zs                                                 0.00361 (0.0124)                                      
f1_xs                                                 -0.0150 (0.0134)                                      
f1_pm                                                                                        0.0241 (0.0185)
f1_zm              

***
##### 2.5 CAMINHO A — Qualificação de Regime, Cenários de Capacidade e Fronteira Consolidada vs. Expansão (Etapas 12, 13 e 16A)

Esta seção explora o **Caminho A** aprovado pelo autor, lendo diretamente de `2_INTI/xdata/` sem duplicar arquivos:
1. **A.1 Quebra estrutural de regime (pós-2005 e pós-2006):** substitui a tendência linear (`zs_tr`) por quebras de nível (`aptidão × pós-2005/2006` e `amz × pós-2005/2006`), mostrando que a queda abrupta da conversão nos municípios de baixa aptidão após 2005 absorve integralmente o coeficiente de `xs` e `xm`.
2. **A.2 Cenários alternativos de capacidade (`x1_psoja-pmilho/exposicoes_alternativas.csv`):** compara o rendimento potencial GAEZ em **alto insumo (`yxHr0`)** com o de **baixo insumo (`yxLr0`)**, com os índices de aptidão (`sxHr0`, `sxLr0`) e com a produtividade histórica realizada na PAM em 1991–2000 (`pam_1991_2000_min7`). O efeito positivo ocorre exclusivamente no rendimento potencial de **alto insumo (`yxHr0`)**, coerente com a tecnologia de pacotes químicos/cultivares em Mato Grosso (`Spera2017`).
3. **A.3 Aptidão biofísica (GAEZ, sinal $+$) vs. Ocupação agrícola pré-determinada (MapBiomas 1985–2000, sinal $-$) e mecanismo de intensificação:**
   - Constrói a participação municipal média de agricultura em 1985–2000 (classes 20, 39, 41, 62) e de soja em 2000 (classe 39) a partir de `h1_h2__mapbiomas_classes__col10_1__municipio_bioma_anual.csv`.
   - Demonstra que a ocupação agrícola pré-2001 é **ortogonal à aptidão GAEZ** ($r = -0,025$), multiplica por quase 4 o $R^2$ within (para `0,0754`) e tem **sinal negativo (`-0,116***` a `-0,146***`)** sobre a abertura de vegetação nativa, ao mesmo tempo em que se associa positivamente à **expansão da área plantada de milho de 2ª safra (`+2,51***` pp/década e `+0,57*` com preço)** em terras já abertas (`MacedoEtAl2012`; `Braganca2018`).

In [7]:
# A.1 Quebra estrutural de regime (pós-2005 e pós-2006)
ev <- function(crop, pcol) {
  z <- if (crop == "soja") "zs" else "zm"; x <- if (crop == "soja") "xs" else "xm"
  d <- d_obs %>% mutate(p = .data[[pcol]], e_amz = amz * p, e_z = .data[[z]] * p, e_zamz = .data[[z]] * amz * p)
  m <- feols(as.formula(paste("y_agr ~", x, "+ e_amz + e_z + e_zamz + rain | ibge + ano")), d, vcov = ~ibge)
  rec(m, paste0("E_", pcol, "_", crop), "eventos", paste(crop, "evento", pcol), "obs")
}
cat("=== CAMINHO A.1: Quebra estrutural pós-2005 e pós-2006 ===\n")
tab(ev("soja", "post05"), ev("soja", "post06"), ev("milho", "post05"), ev("milho", "post06"),
    headers = c("soja pós-2005", "soja pós-2006", "milho pós-2005", "milho pós-2006"))

# A.2 Cenários alternativos de capacidade (GAEZ alto vs. baixo insumo, índice sx e PAM 1991-2000)
x1_path <- file.path(root, "xdata/chap1/processed/harmonizacao/x1_psoja-pmilho/exposicoes_alternativas.csv")
x1_alt <- fread(x1_path, showProgress = FALSE) %>%
  as_tibble() %>%
  select(ibge_int = codigo_ibge, ano, produto, cenario_capacidade, capacidade_z) %>%
  distinct() %>%
  mutate(key = paste0(cenario_capacidade, "_", produto)) %>%
  select(ibge_int, ano, key, capacidade_z) %>%
  pivot_wider(names_from = key, values_from = capacidade_z)

# A.3 Exposição pré-determinada MapBiomas 1985-2000 (classes 20, 39, 41, 62 = agricultura; 39 = soja; 15 = pastagem)
mb_path <- file.path(root, "xdata/chap1/processed/tabular/h1_h2__mapbiomas_classes__col10_1__municipio_bioma_anual.csv")
mb_pre <- fread(mb_path, showProgress = FALSE) %>%
  as_tibble() %>%
  filter(ano >= 1985, ano <= 2000) %>%
  group_by(ibge_int = codigo_ibge, ano) %>%
  summarise(
    tot_ha  = sum(area_ha),
    agr_ha  = sum(area_ha[class_id %in% c(20, 39, 41, 62)]),
    soja_ha = sum(area_ha[class_id == 39]),
    pas_ha  = sum(area_ha[class_id == 15]),
    .groups = "drop"
  ) %>%
  group_by(ibge_int) %>%
  summarise(
    area_mun_ha  = first(tot_ha),
    agr_sh_85_00 = mean(agr_ha / tot_ha),
    pas_sh_85_00 = mean(pas_ha / tot_ha),
    soja_sh_2000 = (soja_ha / tot_ha)[ano == 2000],
    .groups = "drop"
  ) %>%
  mutate(
    z_exp_agr  = (agr_sh_85_00 - mean(agr_sh_85_00)) / sd_pop(agr_sh_85_00),
    z_exp_pas  = (pas_sh_85_00 - mean(pas_sh_85_00)) / sd_pop(pas_sh_85_00),
    z_exp_soja = (soja_sh_2000 - mean(soja_sh_2000)) / sd_pop(soja_sh_2000)
  )
stopifnot(nrow(mb_pre) == 141, !anyNA(mb_pre))

d_camA <- d_obs %>%
  left_join(x1_alt, by = c("ibge_int", "ano")) %>%
  left_join(mb_pre, by = "ibge_int") %>%
  mutate(
    # Interações com cenários alternativos de capacidade (soja)
    xs_yxLr  = lps * gaez_yxLr0_soy_soja,  zs_yxLr_tr  = gaez_yxLr0_soy_soja * trend,
    xs_sxHr  = lps * gaez_sxHr0_soy_soja,  zs_sxHr_tr  = gaez_sxHr0_soy_soja * trend,
    xs_sxLr  = lps * gaez_sxLr0_soy_soja,  zs_sxLr_tr  = gaez_sxLr0_soy_soja * trend,
    xs_pam90 = lps * pam_1991_2000_min7_soja, zs_pam90_tr = pam_1991_2000_min7_soja * trend,
    # Interações com ocupação agrícola pré-determinada (1985-2000 e soja 2000)
    x_exp_agr_s  = lps * z_exp_agr,  z_exp_agr_tr  = z_exp_agr * trend,
    x_exp_soja_s = lps * z_exp_soja, z_exp_soja_tr = z_exp_soja * trend,
    x_exp_agr_m  = lpm * z_exp_agr,
    # Área plantada (PAM) como % da área municipal para testar intensificação em terras já abertas
    soja_plant_pct  = 100 * area_soja_ha / area_mun_ha,
    milho_plant_pct = 100 * area_milho_ha / area_mun_ha
  )

CA_yxHr <- M1
CA_yxLr <- rec(feols(y_agr ~ xs_yxLr + zs_yxLr_tr + rain | ibge + bioma^ano, d_camA, vcov = ~ibge), "CA_yxLr_soja", "caminho_A", "soja GAEZ baixo insumo yxLr0", "obs")
CA_sxHr <- rec(feols(y_agr ~ xs_sxHr + zs_sxHr_tr + rain | ibge + bioma^ano, d_camA, vcov = ~ibge), "CA_sxHr_soja", "caminho_A", "soja GAEZ indice aptidao sxHr0", "obs")
CA_pam  <- rec(feols(y_agr ~ xs_pam90 + zs_pam90_tr + rain | ibge + bioma^ano, d_camA, vcov = ~ibge), "CA_pam90_soja", "caminho_A", "soja PAM rendimento 1991-2000", "obs PAM>=7a")
cat("\n=== CAMINHO A.2: Comparação entre cenários de capacidade (Soja) ===\n")
tab(CA_yxHr, CA_yxLr, CA_sxHr, CA_pam, headers = c("GAEZ yx alto (M1)", "GAEZ yx baixo", "GAEZ sx alto", "PAM 1991-2000"))

EX1 <- rec(feols(y_agr ~ x_exp_agr_s + z_exp_agr_tr + rain | ibge + bioma^ano, d_camA, vcov = ~ibge), "EX_agr_soja", "exposicao_pre", "soja: preco x share agr 1985-2000", "obs")
EX2 <- rec(feols(y_agr ~ x_exp_soja_s + z_exp_soja_tr + rain | ibge + bioma^ano, d_camA, vcov = ~ibge), "EX_soja2000", "exposicao_pre", "soja: preco x share soja 2000", "obs")
EX3 <- rec(feols(y_agr ~ xs + x_exp_agr_s + zs_tr + z_exp_agr_tr + rain | ibge + bioma^ano, d_camA, vcov = ~ibge), "EX_ambos_soja", "exposicao_pre", "soja: GAEZ + share agr 1985-2000", "obs")
EX4 <- rec(feols(y_agr ~ xm + x_exp_agr_m + zm_tr + z_exp_agr_tr + rain | ibge + bioma^ano, d_camA, vcov = ~ibge), "EX_ambos_milho", "exposicao_pre", "milho: GAEZ + share agr 1985-2000", "obs")
EX5 <- rec(feols(milho_plant_pct ~ xm + x_exp_agr_m + zm_tr + z_exp_agr_tr + rain | ibge + bioma^ano, d_camA, vcov = ~ibge), "EX_intens_milho", "exposicao_pre", "Y = % area plantada milho (2a safra)", "obs")
cat(sprintf("\nCorrelações municipais (141) com aptidão GAEZ soja (zs): share_agr_85_00 = %.3f | share_soja_2000 = %.3f\n",
            cor(mun_v2$apt_s, mb_pre$agr_sh_85_00[match(mun_v2$ibge_int, mb_pre$ibge_int)]),
            cor(mun_v2$apt_s, mb_pre$soja_sh_2000[match(mun_v2$ibge_int, mb_pre$ibge_int)])))
cat("=== CAMINHO A.3: Potencial GAEZ (+) vs. Fronteira Agrícola Consolidada em 1985-2000 (-) e Intensificação ===\n")
tab(EX1, EX2, EX3, EX4, EX5, headers = c("Y=conv: agr 85-00", "Y=conv: soja 2000", "Y=conv: GAEZ+agr (S)", "Y=conv: GAEZ+agr (M)", "Y=% área milho"))

=== CAMINHO A.1: Quebra estrutural pós-2005 e pós-2006 ===


                   soja pós-2005      soja pós-2006      milho pós-2005      milho pós-2006
xs            0.000934 (0.00969)   0.00775 (0.0130)                                        
e_amz            -0.0382 (0.115)    -0.0108 (0.103)      -0.123 (0.122)      -0.100 (0.108)
e_z             0.0954* (0.0514)   0.104** (0.0496)   0.104*** (0.0329)   0.114*** (0.0326)
e_zamz            0.0721 (0.123)     0.0373 (0.111)      0.0715 (0.154)      0.0346 (0.142)
rain          -0.0104* (0.00540) -0.0103* (0.00546) -0.0117** (0.00573) -0.0118** (0.00570)
xm                                                    0.00318 (0.00760)   -0.0109 (0.00684)
N                           2769               2769                2769                2769
R2 total                  0.3770             0.3765              0.3742              0.3741
R2 within                 0.0359             0.0352              0.0317              0.0314
Efeitos fixos         ibge + ano         ibge + ano          ibge + ano         


=== CAMINHO A.2: Comparação entre cenários de capacidade (Soja) ===


                GAEZ yx alto (M1)       GAEZ yx baixo        GAEZ sx alto       PAM 1991-2000
xs              0.0848** (0.0358)                                                            
zs_tr          0.0405*** (0.0144)                                                            
rain          -0.0135** (0.00626) -0.0167** (0.00681) -0.0157** (0.00637) -0.0305*** (0.0106)
xs_yxLr                              -0.0191 (0.0286)                                        
zs_yxLr_tr                           0.00487 (0.0125)                                        
xs_sxHr                                                -0.0874** (0.0354)                    
zs_sxHr_tr                                             -0.0337** (0.0146)                    
xs_pam90                                                                     -0.0644 (0.0488)
zs_pam90_tr                                                                 -0.0427* (0.0218)
N                            2767                2767       


Correlações municipais (141) com aptidão GAEZ soja (zs): share_agr_85_00 = -0.025 | share_soja_2000 = 0.014


=== CAMINHO A.3: Potencial GAEZ (+) vs. Fronteira Agrícola Consolidada em 1985-2000 (-) e Intensificação ===


                Y=conv: agr 85-00   Y=conv: soja 2000 Y=conv: GAEZ+agr (S) Y=conv: GAEZ+agr (M)   Y=% área milho
x_exp_agr_s    -0.116*** (0.0429)                        -0.104** (0.0404)                                      
z_exp_agr_tr  -0.0695*** (0.0224)                      -0.0654*** (0.0234)  -0.0696*** (0.0258)  2.51*** (0.583)
rain          -0.0150** (0.00606) -0.0135** (0.00557)  -0.0136** (0.00591)  -0.0147** (0.00612) 0.437*** (0.138)
x_exp_soja_s                       -0.146*** (0.0431)                                                           
z_exp_soja_tr                     -0.0816*** (0.0199)                                                           
xs                                                        0.0496* (0.0269)                                      
zs_tr                                                      0.0188 (0.0137)                                      
xm                                                                             0.00706 (0.0168) 

***
##### 2.6 CAMINHO B1 — Logística, Distância Geodésica aos Portos e Choques de Infraestrutura (Etapa 14A)

Construído diretamente da malha oficial `2_INTI/xdata/chap1/raw/vector/ibge/MT_Municipios_2023.zip` (`EPSG:4674`, 141 municípios):
- **Distâncias geodésicas municipais (centroide ao ponto de exportação/escoamento):**
  - Eixo Sul/Sudeste: Porto de Santos (`-23.9619, -46.3042`) e Porto de Paranaguá (`-25.5047, -48.5106`), mediana de `1.536 km`.
  - Eixo Arco Norte (BR-163): terminais hidroviários de Miritituba/Itaituba (`-4.2690, -55.9750`), mediana de `1.124 km`.
  - Ferrovia em Mato Grosso: Terminal Ferroviário de Rondonópolis (`-16.5833, -54.7167`), inaugurado em **2013** (mediana de `439 km`).
- **Variáveis logísticas no painel:**
  - `prox_sul_z`: proximidade padronizada aos portos do Sul (`-z(min(Santos, Paranaguá))`), interagida com `xs`/`xm` (`xs_proxsul` / `xm_proxsul`);
  - `dist_min_t_100km`: distância mínima efetiva ao porto ativo no ano $t$ (em 100 km), igual a `min(Santos, Paranaguá)` até 2013 e `min(Santos, Paranaguá, Miritituba)` a partir de **2014** (entrada em operação comercial dos terminais privados de Miritituba/Barcarena no Arco Norte);
  - `choc_ferrov_13`: `prox_rond_z * (ano >= 2013)` (choque de inauguração do terminal ferroviário de Rondonópolis em 2013).

In [8]:
zip_path <- normalizePath(file.path(root, "xdata/chap1/raw/vector/ibge/MT_Municipios_2023.zip"), winslash = "/")
shp_mt <- st_read(paste0("/vsizip/", zip_path), quiet = TRUE)
stopifnot(nrow(shp_mt) == 141)
cent_mt <- st_centroid(st_geometry(shp_mt))

pts_log <- st_sfc(
  st_point(c(-46.3042, -23.9619)), # Porto de Santos (SP)
  st_point(c(-48.5106, -25.5047)), # Porto de Paranaguá (PR)
  st_point(c(-55.9750,  -4.2690)), # Miritituba / Itaituba (PA - Arco Norte)
  st_point(c(-54.7167, -16.5833)), # Terminal Ferroviário de Rondonópolis (MT - 2013)
  crs = st_crs(shp_mt)
)
dmat <- as.numeric(st_distance(cent_mt, pts_log)) / 1000
dim(dmat) <- c(nrow(shp_mt), 4)

dist_mun <- tibble(
  ibge_int      = as.integer(shp_mt$CD_MUN),
  dist_sul_km   = pmin(dmat[, 1], dmat[, 2]),
  dist_mirit_km = dmat[, 3],
  dist_rond_km  = dmat[, 4]
) %>%
  mutate(
    prox_sul_z  = -(dist_sul_km - mean(dist_sul_km)) / sd_pop(dist_sul_km),
    prox_rond_z = -(dist_rond_km - mean(dist_rond_km)) / sd_pop(dist_rond_km)
  )

d_camB <- d_obs %>%
  left_join(dist_mun, by = "ibge_int") %>%
  mutate(
    dist_min_t_100km = ifelse(ano <= 2013, dist_sul_km, pmin(dist_sul_km, dist_mirit_km)) / 100,
    x_ps_proxsul     = lps * prox_sul_z,
    xs_proxsul       = xs * prox_sul_z,
    x_pm_proxsul     = lpm * prox_sul_z,
    xm_proxsul       = xm * prox_sul_z,
    choc_ferrov_13   = prox_rond_z * as.numeric(ano >= 2013)
  )

cat("Resumo das distâncias geodésicas municipais (km) nos 141 municípios:\n")
print(summary(dist_mun %>% select(dist_sul_km, dist_mirit_km, dist_rond_km)))

L1_s <- rec(feols(y_agr ~ xs + xs_proxsul + x_ps_proxsul + zs_tr + rain | ibge + bioma^ano, d_camB, vcov = ~ibge), "B1_prox_soja", "caminho_B_logistica", "soja x proximidade porto Sul", "obs")
L2_s <- rec(feols(y_agr ~ xs + dist_min_t_100km + choc_ferrov_13 + zs_tr + rain | ibge + bioma^ano, d_camB, vcov = ~ibge), "B1_choq_soja", "caminho_B_logistica", "soja + Arco Norte 2014 e Ferrovia 2013", "obs")
L1_m <- rec(feols(y_agr ~ xm + xm_proxsul + x_pm_proxsul + zm_tr + rain | ibge + bioma^ano, d_camB, vcov = ~ibge), "B1_prox_milho", "caminho_B_logistica", "milho x proximidade porto Sul", "obs")
L2_m <- rec(feols(y_agr ~ xm + dist_min_t_100km + choc_ferrov_13 + zm_tr + rain | ibge + bioma^ano, d_camB, vcov = ~ibge), "B1_choq_milho", "caminho_B_logistica", "milho + Arco Norte 2014 e Ferrovia 2013", "obs")
cat("\n=== CAMINHO B1: Modelos com Distância aos Portos, Arco Norte (2014) e Ferrovia de Rondonópolis (2013) ===\n")
tab(L1_s, L2_s, L1_m, L2_m, headers = c("Soja × Prox. Sul", "Soja + Choques Log.", "Milho × Prox. Sul", "Milho + Choques Log."))

Resumo das distâncias geodésicas municipais (km) nos 141 municípios:


  dist_sul_km     dist_mirit_km     dist_rond_km     
 Min.   : 986.3   Min.   : 539.4   Min.   :   3.853  
 1st Qu.:1344.8   1st Qu.: 867.2   1st Qu.: 241.969  
 Median :1519.5   Median :1124.1   Median : 439.126  
 Mean   :1519.0   Mean   :1070.7   Mean   : 450.354  
 3rd Qu.:1680.2   3rd Qu.:1267.2   3rd Qu.: 628.743  
 Max.   :2191.2   Max.   :1535.5   Max.   :1011.533  



=== CAMINHO B1: Modelos com Distância aos Portos, Arco Norte (2014) e Ferrovia de Rondonópolis (2013) ===


                    Soja × Prox. Sul Soja + Choques Log.   Milho × Prox. Sul Milho + Choques Log.
xs                  0.101** (0.0411)   0.0940** (0.0394)                                         
xs_proxsul           0.0486 (0.0323)                                                             
x_ps_proxsul       -0.00474 (0.0541)                                                             
zs_tr             0.0405*** (0.0144)  0.0599*** (0.0157)                                         
rain             -0.0136** (0.00624) -0.0149** (0.00625) -0.0155** (0.00677)  -0.0173** (0.00687)
dist_min_t_100km                     0.0252*** (0.00789)                      0.0261*** (0.00841)
choc_ferrov_13                        -0.169*** (0.0453)                       -0.128*** (0.0395)
xm                                                           0.0230 (0.0179)   0.0656*** (0.0214)
xm_proxsul                                                   0.0604 (0.0410)                     
x_pm_proxsul        

***
##### 2.7 CAMINHO B2 — Embargos do IBAMA Geocodificados (`termo_embargo_historico.csv`) e Medidas Alternativas de Fiscalização (Etapa 14D)

**Por que usar os embargos geocodificados em vez de apenas `autos_flora`?**
- `x3_n_autos_flora_elegiveis_lag1__ibama_espacial` exige classificação temática completa na fonte, o que reduz a amostra de 2.772 para 1.140 observações (ou introduz 48% de zeros administrativos).
- Lendo diretamente `2_INTI/xdata/chap1/raw/tabular/ibama/termo_embargo_historico.csv`, filtramos os registros de Mato Grosso não cancelados (`SIT_CANCELADO != "S"`), deduplicamos por termo único (`SEQ_TAD`) mantendo a data original do embargo (`DAT_EMBARGO`) e cruzamos as coordenadas (`NUM_LONGITUDE_TAD`, `NUM_LATITUDE_TAD`) com os polígonos oficiais de `MT_Municipios_2023.zip`:
  - **12.283 termos únicos de embargo** entre 2000 e 2024 em MT (100% com código IBGE válido nos 141 municípios e **90,5% com coordenadas válidas geocodificadas por ponto-no-polígono**).
  - Isso permite estimar a dimensão institucional **em toda a amostra observada (`obs = TRUE`, $N = 2.767$, 137 municípios)**, tanto pelo número de embargos geocodificados em $t-1$ (`emb_esp_l1 = ln(1 + n_emb_esp_{t-1})`) quanto pela **área embargada em hectares em $t-1$** (`emb_area_l1 = ln(1 + area_emb_esp_ha_{t-1})`).
  - *Nota metodológica:* conforme registrado em `2_INTI/xdata/chap1/processed/harmonizacao/x3e_infracoes/pendencias.csv`, a série de embargos tem atualizações cadastrais na fonte; por isso deduplicamos estritamente pelo primeiro registro temporal de cada `SEQ_TAD` não cancelado e comparamos também com `n_autos_desmatamento_lag1` de `x3b_infracoes/instituicoes_espacial.csv`.

In [9]:
emb_path <- file.path(root, "xdata/chap1/raw/tabular/ibama/termo_embargo_historico.csv")
dt_emb <- fread(
  emb_path, sep = ";",
  select = c("SEQ_TAD", "SIT_CANCELADO", "DAT_EMBARGO", "COD_MUNICIPIO", "UF",
             "NUM_LONGITUDE_TAD", "NUM_LATITUDE_TAD", "QTD_AREA_EMBARGADA"),
  encoding = "UTF-8", showProgress = FALSE
)

dt_mt <- dt_emb[(UF == "MT" | substr(as.character(COD_MUNICIPIO), 1, 2) == "51") & SIT_CANCELADO != "S"]
dt_mt[, dt := as.Date(substr(DAT_EMBARGO, 1, 10))]
dt_mt[, yr := as.integer(format(dt, "%Y"))]
dt_mt <- dt_mt[!is.na(yr) & yr >= 2000 & yr <= 2024]
setorder(dt_mt, SEQ_TAD, dt)
dedup_emb <- unique(dt_mt, by = "SEQ_TAD")
dedup_emb[, area_ha := suppressWarnings(as.numeric(gsub(",", ".", QTD_AREA_EMBARGADA)))]
dedup_emb[is.na(area_ha) | area_ha < 0, area_ha := 0]

# Geocodificação ponto-no-polígono na malha IBGE 2023 (141 municípios)
dedup_emb[, lon := suppressWarnings(as.numeric(gsub(",", ".", NUM_LONGITUDE_TAD)))]
dedup_emb[, lat := suppressWarnings(as.numeric(gsub(",", ".", NUM_LATITUDE_TAD)))]
pts_valid <- dedup_emb[!is.na(lon) & !is.na(lat) & lon >= -65 & lon <= -48 & lat >= -20 & lat <= -6]
sf_emb <- st_as_sf(pts_valid, coords = c("lon", "lat"), crs = st_crs(shp_mt))
joined_emb <- st_join(sf_emb, shp_mt["CD_MUN"], join = st_within)
st_geometry(joined_emb) <- NULL

grid_emb <- CJ(ibge_int = dist_mun$ibge_int, yr = 2000:2024)
emb_esp <- as.data.table(joined_emb)[!is.na(CD_MUN), .(
  n_emb_esp = .N,
  area_emb_esp_ha = sum(area_ha)
), by = .(ibge_int = as.integer(CD_MUN), yr)]

emb_panel <- merge(grid_emb, emb_esp, by = c("ibge_int", "yr"), all.x = TRUE)
emb_panel[is.na(n_emb_esp), `:=`(n_emb_esp = 0L, area_emb_esp_ha = 0)]
setorder(emb_panel, ibge_int, yr)
emb_panel[, `:=`(
  emb_esp_l1  = log1p(shift(n_emb_esp, 1L)),
  emb_area_l1 = log1p(shift(area_emb_esp_ha, 1L))
), by = ibge_int]

# Leitura complementar de autos de desmatamento de x3b_infracoes/instituicoes_espacial.csv
x3b_path <- file.path(root, "xdata/chap1/processed/harmonizacao/x3b_infracoes/instituicoes_espacial.csv")
x3b_desm <- fread(x3b_path, showProgress = FALSE) %>%
  as_tibble() %>%
  transmute(
    ibge_int = as.integer(codigo_ibge),
    ano = as.integer(ano),
    f_desm_l1 = log1p(n_autos_desmatamento_lag1)
  )

d_camB <- d_camB %>%
  left_join(as_tibble(emb_panel) %>% rename(ano = yr), by = c("ibge_int", "ano")) %>%
  left_join(x3b_desm, by = c("ibge_int", "ano")) %>%
  mutate(
    emb_ps = lps * emb_esp_l1, emb_zs = zs * emb_esp_l1, emb_xs = xs * emb_esp_l1,
    emb_area_xs = xs * emb_area_l1,
    emb_area_xm = xm * emb_area_l1
  )

cat(sprintf("Termos de embargo únicos não cancelados em MT (2000-2024): %d | Geocodificados na malha 2023: %d (%.1f%%)\n",
            nrow(dedup_emb), nrow(pts_valid), 100 * nrow(pts_valid) / nrow(dedup_emb)))

B2_s1 <- rec(feols(y_agr ~ xs + emb_esp_l1 + emb_ps + emb_zs + emb_xs + zs_tr + rain | ibge + bioma^ano, d_camB, vcov = ~ibge),
             "B2_emb_mod_soja", "caminho_B_embargos", "soja + moderacao embargos geocodificados", "obs")
B2_s2 <- rec(feols(y_agr ~ xs + emb_area_l1 + emb_area_xs + zs_tr + rain | ibge + bioma^ano, d_camB, vcov = ~ibge),
             "B2_emb_area_soja", "caminho_B_embargos", "soja + area embargada (ha) lag1", "obs")
B2_m2 <- rec(feols(y_agr ~ xm + emb_area_l1 + emb_area_xm + zm_tr + rain | ibge + bioma^ano, d_camB, vcov = ~ibge),
             "B2_emb_area_milho", "caminho_B_embargos", "milho + area embargada (ha) lag1", "obs")
B2_desm <- rec(feols(y_agr ~ xs + f_desm_l1 + zs_tr + rain | ibge + bioma^ano, d_camB, vcov = ~ibge),
               "B2_autos_desm_soja", "caminho_B_embargos", "soja + autos desmatamento lag1", "obs c/ desm")

cat("\n=== CAMINHO B2: Embargos do IBAMA Geocodificados (N = 2.767) e Autos de Desmatamento ===\n")
tab(B2_s1, B2_s2, B2_m2, B2_desm,
    headers = c("Soja × N Embargos", "Soja + Área Embarg.", "Milho + Área Embarg.", "Soja + Autos Desm."))

Termos de embargo únicos não cancelados em MT (2000-2024): 12284 | Geocodificados na malha 2023: 11513 (93.7%)



=== CAMINHO B2: Embargos do IBAMA Geocodificados (N = 2.767) e Autos de Desmatamento ===


                Soja × N Embargos   Soja + Área Embarg.  Milho + Área Embarg.  Soja + Autos Desm.
xs              0.0933** (0.0375)     0.0835** (0.0361)                           0.0778 (0.0618)
emb_esp_l1       0.00821 (0.0124)                                                                
emb_ps           -0.0457 (0.0323)                                                                
emb_zs           0.00774 (0.0171)                                                                
emb_xs          -0.0314* (0.0179)                                                                
zs_tr          0.0389*** (0.0139)    0.0396*** (0.0149)                          0.0345* (0.0203)
rain          -0.0137** (0.00620)   -0.0138** (0.00633)   -0.0154** (0.00678) -0.0120** (0.00547)
emb_area_l1                       -0.00755*** (0.00285) -0.00820*** (0.00298)                    
emb_area_xs                          0.000106 (0.00950)                                          
xm                  

***
##### 2.8 Cortes Temporais por Marcos Institucionais (`2001–2005`, `2006–2011`, `2012–2024`) $\times$ Biomas (Amazônia vs. Cerrado) e Magnitudes Econômicas

Decompomos a interação principal ($\ln P_{c,t-1} \times Z_{i,c}$) nos três regimes histórico-institucionais destacados no artigo, cruzando-os com os biomas predominantes (**Amazônia** vs. **Cerrado**):
- **Regime P1 (`2001–2005`):** Entrada da China na OMC (dez/2001) e expansão agrícola anterior aos marcos de contenção;
- **Regime P2 (`2006–2011`):** Implementação do PPCDAm e da Moratória da Soja na Amazônia (jul/2006) até a véspera do Novo Código Florestal;
- **Regime P3 (`2012–2024`):** Vigência do Novo Código Florestal (Lei nº 12.651, maio/2012) e expansão logística (Ferrovia de Rondonópolis em 2013 e Arco Norte em 2014).

Também documentamos a decomposição do poder explicativo ($R^2$ total vs. $R^2$ *within*) e as magnitudes econômicas práticas para choques de 1 desvio-padrão e de ciclo completo de preços.

In [10]:
d_reg <- d_obs %>% mutate(
  d_p1 = as.numeric(ano <= 2005),
  d_p2 = as.numeric(ano >= 2006 & ano <= 2011),
  d_p3 = as.numeric(ano >= 2012),
  xs_p1 = xs * d_p1, xs_p2 = xs * d_p2, xs_p3 = xs * d_p3,
  xm_p1 = xm * d_p1, xm_p2 = xm * d_p2, xm_p3 = xm * d_p3
)

RP_s_mt  <- rec(feols(y_agr ~ xs_p1 + xs_p2 + xs_p3 + rain | ibge + bioma^ano, d_reg, vcov = ~ibge),
                'REG_prc_soja_MT', 'regimes_biomas', 'Preços Soja: 3 regimes MT', 'obs')
RP_s_amz <- rec(feols(y_agr ~ xs_p1 + xs_p2 + xs_p3 + rain | ibge + ano, filter(d_reg, bioma == bioma_amz), vcov = ~ibge),
                'REG_prc_soja_Amz', 'regimes_biomas', 'Preços Soja: 3 regimes Amazônia', 'obs Amazônia')
RP_s_cer <- rec(feols(y_agr ~ xs_p1 + xs_p2 + xs_p3 + rain | ibge + ano, filter(d_reg, bioma == bioma_cer), vcov = ~ibge),
                'REG_prc_soja_Cer', 'regimes_biomas', 'Preços Soja: 3 regimes Cerrado', 'obs Cerrado')
RP_m_mt  <- rec(feols(y_agr ~ xm_p1 + xm_p2 + xm_p3 + rain | ibge + bioma^ano, d_reg, vcov = ~ibge),
                'REG_prc_milho_MT', 'regimes_biomas', 'Preços Milho: 3 regimes MT', 'obs')
RP_m_amz <- rec(feols(y_agr ~ xm_p1 + xm_p2 + xm_p3 + rain | ibge + ano, filter(d_reg, bioma == bioma_amz), vcov = ~ibge),
                'REG_prc_milho_Amz', 'regimes_biomas', 'Preços Milho: 3 regimes Amazônia', 'obs Amazônia')
RP_m_cer <- rec(feols(y_agr ~ xm_p1 + xm_p2 + xm_p3 + rain | ibge + ano, filter(d_reg, bioma == bioma_cer), vcov = ~ibge),
                'REG_prc_milho_Cer', 'regimes_biomas', 'Preços Milho: 3 regimes Cerrado', 'obs Cerrado')

cat('=== REGIMES TEMPORAIS (2001-05 | 2006-11 | 2012-24) × BIOMAS — H1-PREÇOS ===\n')
tab(RP_s_mt, RP_s_amz, RP_s_cer, RP_m_mt, RP_m_amz, RP_m_cer,
    headers = c('Soja MT', 'Soja Amazônia', 'Soja Cerrado', 'Milho MT', 'Milho Amazônia', 'Milho Cerrado'))

# Testes de igualdade de inclinações entre regimes (P1 vs P2 e P1 vs P3)
cat('\n=== Testes lineares de mudança de regime (P1 vs P2 e P1 vs P3) ===\n')
for (nm in list(list(m = RP_s_mt, v1 = 'xs_p1', v2 = 'xs_p2', v3 = 'xs_p3', lbl = 'Soja MT'),
                list(m = RP_s_amz, v1 = 'xs_p1', v2 = 'xs_p2', v3 = 'xs_p3', lbl = 'Soja Amazônia'),
                list(m = RP_s_cer, v1 = 'xs_p1', v2 = 'xs_p2', v3 = 'xs_p3', lbl = 'Soja Cerrado'),
                list(m = RP_m_mt, v1 = 'xm_p1', v2 = 'xm_p2', v3 = 'xm_p3', lbl = 'Milho MT'),
                list(m = RP_m_amz, v1 = 'xm_p1', v2 = 'xm_p2', v3 = 'xm_p3', lbl = 'Milho Amazônia'),
                list(m = RP_m_cer, v1 = 'xm_p1', v2 = 'xm_p2', v3 = 'xm_p3', lbl = 'Milho Cerrado'))) {
  b <- coef(nm$m); V <- vcov(nm$m)
  d12 <- b[nm$v1] - b[nm$v2]; se12 <- sqrt(V[nm$v1, nm$v1] + V[nm$v2, nm$v2] - 2 * V[nm$v1, nm$v2])
  p12 <- 2 * pt(abs(d12 / se12), df = degrees_freedom(nm$m, 't'), lower.tail = FALSE)
  d13 <- b[nm$v1] - b[nm$v3]; se13 <- sqrt(V[nm$v1, nm$v1] + V[nm$v3, nm$v3] - 2 * V[nm$v1, nm$v3])
  p13 <- 2 * pt(abs(d13 / se13), df = degrees_freedom(nm$m, 't'), lower.tail = FALSE)
  cat(sprintf('%-15s | P1-P2 = %+.4f (EP = %.4f, p = %.4f) | P1-P3 = %+.4f (EP = %.4f, p = %.4f)\n',
              nm$lbl, d12, se12, p12, d13, se13, p13))
}

# Resumo de médias por regime e magnitudes econômicas práticas
cat('\n=== Médias e desvios-padrão de Y_agr (p.p./ano) por regime na amostra de referência ===\n')
d_ref <- d_reg[fixest::obs(M1), ]
print(d_ref %>% mutate(regime = case_when(ano <= 2005 ~ '1_2001-2005', ano <= 2011 ~ '2_2006-2011', TRUE ~ '3_2012-2024')) %>%
      group_by(regime) %>% summarise(n = n(), media_pp = mean(y_agr), dp_pp = sd(y_agr), .groups = 'drop'))
cat(sprintf('Painel de referência (N=%d): media_pp = %.4f | dp_pp = %.4f\n', nrow(d_ref), mean(d_ref$y_agr), sd(d_ref$y_agr)))
cat(sprintf('Dispersão de ln P (24 anos): DP(ln Ps) = %.4f (amplitude = %.4f) | DP(ln Pm) = %.4f (amplitude = %.4f)\n',
            sd(anos_v2 %>% mutate(lps = log(ps_l1)) %>% pull(lps)),
            diff(range(log(anos_v2$ps_l1))),
            sd(anos_v2 %>% mutate(lpm = log(pm_l1)) %>% pull(lpm)),
            diff(range(log(anos_v2$pm_l1)))))

=== REGIMES TEMPORAIS (2001-05 | 2006-11 | 2012-24) × BIOMAS — H1-PREÇOS ===


                          Soja MT      Soja Amazônia       Soja Cerrado            Milho MT     Milho Amazônia      Milho Cerrado
xs_p1            0.354*** (0.135)      0.400 (0.246)    0.339** (0.163)                                                          
xs_p2             0.0108 (0.0211) -0.0643** (0.0318)  0.0523** (0.0250)                                                          
xs_p3             0.0120 (0.0111)   0.00965 (0.0153)   0.00566 (0.0149)                                                          
rain          -0.0138** (0.00618) -0.0231** (0.0104) -0.00386 (0.00574) -0.0161** (0.00669) -0.0239** (0.0107) -0.00757 (0.00727)
xm_p1                                                                      0.423*** (0.121)      0.367 (0.329)   0.450*** (0.134)
xm_p2                                                                       0.0198 (0.0266)    -0.114 (0.0729)   0.0553* (0.0298)
xm_p3                                                                    -0.00407 (0.00767


=== Testes lineares de mudança de regime (P1 vs P2 e P1 vs P3) ===


Soja MT         | P1-P2 = +0.3433 (EP = 0.1286, p = 0.0085) | P1-P3 = +0.3421 (EP = 0.1347, p = 0.0123)
Soja Amazônia  | P1-P2 = +0.4642 (EP = 0.2633, p = 0.0822) | P1-P3 = +0.3902 (EP = 0.2414, p = 0.1104)
Soja Cerrado    | P1-P2 = +0.2869 (EP = 0.1484, p = 0.0580) | P1-P3 = +0.3336 (EP = 0.1645, p = 0.0471)
Milho MT        | P1-P2 = +0.4036 (EP = 0.1105, p = 0.0004) | P1-P3 = +0.4275 (EP = 0.1218, p = 0.0006)
Milho Amazônia | P1-P2 = +0.4810 (EP = 0.3475, p = 0.1706) | P1-P3 = +0.3850 (EP = 0.3279, p = 0.2443)
Milho Cerrado   | P1-P2 = +0.3946 (EP = 0.1185, p = 0.0015) | P1-P3 = +0.4521 (EP = 0.1359, p = 0.0015)



=== Médias e desvios-padrão de Y_agr (p.p./ano) por regime na amostra de referência ===


# A tibble: 3 × 4
  regime          n media_pp dp_pp
  <chr>       <int>    <dbl> <dbl>
1 1_2001-2005   578   0.301  0.753
2 2_2006-2011   667   0.0615 0.145
3 3_2012-2024  1522   0.0633 0.122


Painel de referência (N=2767): media_pp = 0.1124 | dp_pp = 0.3754


Dispersão de ln P (24 anos): DP(ln Ps) = 0.2446 (amplitude = 0.8517) | DP(ln Pm) = 0.2893 (amplitude = 0.9446)


***
##### 2.8b Extensões da Revisão Técnica (Onda 2): Amostra Efetiva, Regimes Irrestritos, Pontes em Amostra Comum, Embargos e FWL-HAC

Responde às demandas metodológicas da revisão técnica usando exclusivamente os dados já presentes no projeto (`base_modelos_h1_prec.csv` e `2_INTI/xdata/`):
1. **Amostra efetiva exata ($N = 2.767$), balanço incluídas vs. excluídas e painel balanceado (95 municípios):** estatísticas exatas da amostra efetiva de `M1`/`M2` e reestimação no painel balanceado (`24 anos` completos por município).
2. **Regimes institucionais irrestritos ($Z \times D_{2t}$ e $Z \times D_{3t}$), tendência contínua e teste formal entre biomas:** centragem dos preços dentro de cada regime, inclusão das interações de nível `Z × regime`, testes de Wald conjuntos e contraste direto Amazônia vs. Cerrado (`Amz − Cer`).
3. **Teste formal de diferença entre destinos (`y_agr` vs. `y_pas`) por empilhamento (*stacked*):** estimação empilhada na amostra comum ($N = 2.767 \times 2 = 5.534$) para testar $H_0: \beta_{\text{agr}} = \beta_{\text{past}}$.
4. **Pontes em amostra comum:** (i) GAEZ `yxHr0` vs. PAM 1991–2000 ($N = 1.016$); (ii) amostra fiscal ($N = 1.135$) com especificação alinhada (`ibge + bioma^ano + zs_tr`) e comparação direta com embargos; (iii) decomposição passo a passo de `ev` (tendência $\to$ degrau preservando `bioma^ano`, depois `bioma^ano` $\to$ `ano`, depois interações Amazônia); (iv) ponte CONAB ($N = 2.650$).
5. **Embargos do IBAMA:** hierarquia completa na Eq. (4) (`B2_emb_area_full_*`) com testes de Wald conjuntos e gradiente marginal $\hat{\beta} + \hat{\eta}_2 \ln(1+N)$ com IC 95% pelo método delta na Eq. (5).
6. **Projeção FWL exata com HAC na série anual de *scores* (`DK(2)` e FWL-HAC) e verificação de não-linearidade por tercis de $Z$.**


In [11]:
# ==============================================================================
# SEÇÃO 2.8b — EXTENSÕES DA REVISÃO TÉCNICA (ONDA 2)
# ==============================================================================
cat("=== INICIANDO SEÇÃO 2.8b (ONDA 2) ===\n")

# 1) Amostra efetiva exata (N = 2.767), descritivas por regime, balanço e painel balanceado
idx_m1 <- fixest::obs(M1)
stopifnot(length(idx_m1) == nobs(M1), length(idx_m1) == 2767)

raw_extra <- raw_v2 %>%
  transmute(
    ibge_int = as.integer(z0_codigo_ibge),
    ano = as.integer(z0_ano),
    nativa_ini_ha = as.numeric(x1b_y_nativa_inicial_ha),
    conv_agr_ha   = as.numeric(y1a_conversao_ha__agricultura),
    conv_pas_ha   = as.numeric(y1a_conversao_ha__pastagem)
  )

eff_keys <- d_obs[idx_m1, c("ibge_int", "ano")] %>% mutate(in_M1 = TRUE)
d_all_diag <- df_v2 %>%
  left_join(raw_extra, by = c("ibge_int", "ano")) %>%
  left_join(eff_keys, by = c("ibge_int", "ano")) %>%
  mutate(
    in_M1 = replace_na(in_M1, FALSE),
    regime = case_when(ano <= 2005 ~ "1_2001-2005", ano <= 2011 ~ "2_2006-2011", TRUE ~ "3_2012-2024")
  )

d_eff <- d_all_diag %>% filter(in_M1)
stopifnot(nrow(d_eff) == 2767)

sum_eff_reg <- bind_rows(
  d_eff %>% mutate(regime = "0_total_M1") %>%
    group_by(regime) %>%
    summarise(
      n = n(), n_mun = n_distinct(ibge_int),
      media_y_agr = mean(y_agr), dp_y_agr = sd(y_agr), mediana_y_agr = median(y_agr),
      media_y_pas = mean(y_pas), media_nativa_ha = mean(nativa_ini_ha),
      razao_somas_pp = 100 * sum(conv_agr_ha) / sum(nativa_ini_ha),
      .groups = "drop"
    ),
  d_eff %>%
    group_by(regime) %>%
    summarise(
      n = n(), n_mun = n_distinct(ibge_int),
      media_y_agr = mean(y_agr), dp_y_agr = sd(y_agr), mediana_y_agr = median(y_agr),
      media_y_pas = mean(y_pas), media_nativa_ha = mean(nativa_ini_ha),
      razao_somas_pp = 100 * sum(conv_agr_ha) / sum(nativa_ini_ha),
      .groups = "drop"
    )
) %>% mutate(pct_n = 100 * n / nrow(d_eff))

price_sds <- bind_rows(
  anos_v2 %>% summarise(regime = "0_total_24anos", n_anos = n(),
                        dp_ln_ps = sd(log(ps_l1)), amp_ln_ps = diff(range(log(ps_l1))),
                        dp_ln_pm = sd(log(pm_l1)), amp_ln_pm = diff(range(log(pm_l1)))),
  anos_v2 %>% mutate(regime = case_when(ano <= 2005 ~ "1_2001-2005", ano <= 2011 ~ "2_2006-2011", TRUE ~ "3_2012-2024")) %>%
    group_by(regime) %>%
    summarise(n_anos = n(),
              dp_ln_ps = sd(log(ps_l1)), amp_ln_ps = diff(range(log(ps_l1))),
              dp_ln_pm = sd(log(pm_l1)), amp_ln_pm = diff(range(log(pm_l1))),
              .groups = "drop")
)

cat("\n=== Onda 2.1: Amostra efetiva exata (N = 2.767) por regime ===\n")
print(as.data.frame(sum_eff_reg), row.names = FALSE)
cat("\n=== Onda 2.1: Dispersão dos log-preços (24 anos vs. por regime) ===\n")
print(as.data.frame(price_sds), row.names = FALSE)
write.csv(left_join(sum_eff_reg, price_sds, by = "regime"),
          file.path(out_dir, "onda2_amostra_regimes.csv"), row.names = FALSE, fileEncoding = "UTF-8")

# Balanço incluídas (N = 2.767) vs. excluídas (N = 617: 612 preenchidas + 5 singletons FE)
bal_vars <- c("y_agr", "y_pas", "apt_s", "apt_m", "amz", "cer", "pan", "nativa_ini_ha", "area_soja_ha", "area_milho_ha", "rain")
bal_tbl <- map_dfr(bal_vars, function(v) {
  x_in  <- d_all_diag[[v]][d_all_diag$in_M1]
  x_out <- d_all_diag[[v]][!d_all_diag$in_M1]
  m_dif <- feols(as.formula(paste(v, "~ in_M1")), data = d_all_diag, vcov = ~ibge_int)
  ct <- coeftable(m_dif)
  tibble(
    variavel = v,
    n_in = sum(!is.na(x_in)), media_in = mean(x_in, na.rm = TRUE), dp_in = sd(x_in, na.rm = TRUE),
    n_out = sum(!is.na(x_out)), media_out = mean(x_out, na.rm = TRUE), dp_out = sd(x_out, na.rm = TRUE),
    diff_in_minus_out = ct["in_M1TRUE", 1], ep_cluster = ct["in_M1TRUE", 2], p_cluster = ct["in_M1TRUE", 4]
  )
})
cat("\n=== Onda 2.1b: Balanço incluídas (2.767) vs. excluídas (617) ===\n")
print(as.data.frame(bal_tbl), row.names = FALSE)
write.csv(bal_tbl, file.path(out_dir, "onda2_balanco_amostra.csv"), row.names = FALSE, fileEncoding = "UTF-8")

# Painel estritamente balanceado (municípios com 24 anos completos com obs == TRUE)
mun_bal <- d_obs %>% count(ibge_int) %>% filter(n == 24L) %>% pull(ibge_int)
d_bal95 <- d_obs %>% filter(ibge_int %in% mun_bal)
stopifnot(nrow(d_bal95) == 24L * length(mun_bal))
cat(sprintf("\nPainel estritamente balanceado: %d municípios x 24 anos = %d obs.\n", length(mun_bal), nrow(d_bal95)))

BAL_M1 <- run("soja",  d_bal95, "BAL95_M1_soja",  "onda2_balanceado", "M1 soja painel balanceado 95 mun",  "bal95")
BAL_M2 <- run("milho", d_bal95, "BAL95_M2_milho", "onda2_balanceado", "M2 milho painel balanceado 95 mun", "bal95")

# Lista para consolidar testes conjuntos e contrastes da Onda 2
TESTS_O2 <- list()
add_test <- function(grupo, teste, cultura = NA_character_, amostra = NA_character_,
                     est = NA_real_, ep = NA_real_, stat = NA_real_, df1 = NA_real_, df2 = NA_real_, p = NA_real_, nota = NA_character_) {
  TESTS_O2[[length(TESTS_O2) + 1]] <<- tibble(
    grupo = grupo, teste = teste, cultura = cultura, amostra = amostra,
    estimativa = est, erro_padrao = ep, estatistica = stat, df1 = df1, df2 = df2, p = p, nota = nota
  )
}

# 2) Regimes institucionais irrestritos (Z x regime), tendência contínua e diferença Amazônia vs. Cerrado
c_ps_p1 <- mean(log(anos_v2$ps_l1[anos_v2$ano <= 2005]))
c_ps_p2 <- mean(log(anos_v2$ps_l1[anos_v2$ano >= 2006 & anos_v2$ano <= 2011]))
c_ps_p3 <- mean(log(anos_v2$ps_l1[anos_v2$ano >= 2012]))
c_pm_p1 <- mean(log(anos_v2$pm_l1[anos_v2$ano <= 2005]))
c_pm_p2 <- mean(log(anos_v2$pm_l1[anos_v2$ano >= 2006 & anos_v2$ano <= 2011]))
c_pm_p3 <- mean(log(anos_v2$pm_l1[anos_v2$ano >= 2012]))

mk_reg_vars <- function(d) {
  d %>% mutate(
    d_p1 = as.numeric(ano <= 2005),
    d_p2 = as.numeric(ano >= 2006 & ano <= 2011),
    d_p3 = as.numeric(ano >= 2012),
    lps_reg = log(ps_l1) - case_when(ano <= 2005 ~ c_ps_p1, ano <= 2011 ~ c_ps_p2, TRUE ~ c_ps_p3),
    lpm_reg = log(pm_l1) - case_when(ano <= 2005 ~ c_pm_p1, ano <= 2011 ~ c_pm_p2, TRUE ~ c_pm_p3),
    xs_r_p1 = lps_reg * zs * d_p1, xs_r_p2 = lps_reg * zs * d_p2, xs_r_p3 = lps_reg * zs * d_p3,
    zs_p2   = zs * d_p2,           zs_p3   = zs * d_p3,
    xm_r_p1 = lpm_reg * zm * d_p1, xm_r_p2 = lpm_reg * zm * d_p2, xm_r_p3 = lpm_reg * zm * d_p3,
    zm_p2   = zm * d_p2,           zm_p3   = zm * d_p3
  )
}

d_reg_u <- mk_reg_vars(d_obs)
d_bal95_r <- mk_reg_vars(d_bal95)

RUL_s_mt  <- rec(feols(y_agr ~ xs_r_p1 + xs_r_p2 + xs_r_p3 + zs_p2 + zs_p3 + rain | ibge + bioma^ano, d_reg_u, vcov = ~ibge),
                 "REG_lvl_soja_MT", "onda2_regimes_irrestritos", "Soja: 3 regimes irrestritos MT (Z x regime)", "obs")
RUL_s_amz <- rec(feols(y_agr ~ xs_r_p1 + xs_r_p2 + xs_r_p3 + zs_p2 + zs_p3 + rain | ibge + ano, filter(d_reg_u, bioma == bioma_amz), vcov = ~ibge),
                 "REG_lvl_soja_Amz", "onda2_regimes_irrestritos", "Soja: 3 regimes irrestritos Amazonia", "obs Amazonia")
RUL_s_cer <- rec(feols(y_agr ~ xs_r_p1 + xs_r_p2 + xs_r_p3 + zs_p2 + zs_p3 + rain | ibge + ano, filter(d_reg_u, bioma == bioma_cer), vcov = ~ibge),
                 "REG_lvl_soja_Cer", "onda2_regimes_irrestritos", "Soja: 3 regimes irrestritos Cerrado", "obs Cerrado")

RUL_m_mt  <- rec(feols(y_agr ~ xm_r_p1 + xm_r_p2 + xm_r_p3 + zm_p2 + zm_p3 + rain | ibge + bioma^ano, d_reg_u, vcov = ~ibge),
                 "REG_lvl_milho_MT", "onda2_regimes_irrestritos", "Milho: 3 regimes irrestritos MT (Z x regime)", "obs")
RUL_m_amz <- rec(feols(y_agr ~ xm_r_p1 + xm_r_p2 + xm_r_p3 + zm_p2 + zm_p3 + rain | ibge + ano, filter(d_reg_u, bioma == bioma_amz), vcov = ~ibge),
                 "REG_lvl_milho_Amz", "onda2_regimes_irrestritos", "Milho: 3 regimes irrestritos Amazonia", "obs Amazonia")
RUL_m_cer <- rec(feols(y_agr ~ xm_r_p1 + xm_r_p2 + xm_r_p3 + zm_p2 + zm_p3 + rain | ibge + ano, filter(d_reg_u, bioma == bioma_cer), vcov = ~ibge),
                 "REG_lvl_milho_Cer", "onda2_regimes_irrestritos", "Milho: 3 regimes irrestritos Cerrado", "obs Cerrado")

# Sensibilidade com tendência linear + regimes no MT + painel balanceado 95 mun.
RUT_s_mt <- rec(feols(y_agr ~ xs_r_p1 + xs_r_p2 + xs_r_p3 + zs_p2 + zs_p3 + zs_tr + rain | ibge + bioma^ano, d_reg_u, vcov = ~ibge),
                "REG_tr_soja_MT", "onda2_regimes_irrestritos", "Soja: 3 regimes irrestritos + zs_tr MT", "obs")
RUT_m_mt <- rec(feols(y_agr ~ xm_r_p1 + xm_r_p2 + xm_r_p3 + zm_p2 + zm_p3 + zm_tr + rain | ibge + bioma^ano, d_reg_u, vcov = ~ibge),
                "REG_tr_milho_MT", "onda2_regimes_irrestritos", "Milho: 3 regimes irrestritos + zm_tr MT", "obs")

BAL_reg_s <- rec(feols(y_agr ~ xs_r_p1 + xs_r_p2 + xs_r_p3 + zs_p2 + zs_p3 + rain | ibge + bioma^ano, d_bal95_r, vcov = ~ibge),
                 "BAL95_REG_soja", "onda2_balanceado", "Soja: 3 regimes irrestritos bal 95 mun", "bal95")
BAL_reg_m <- rec(feols(y_agr ~ xm_r_p1 + xm_r_p2 + xm_r_p3 + zm_p2 + zm_p3 + rain | ibge + bioma^ano, d_bal95_r, vcov = ~ibge),
                 "BAL95_REG_milho", "onda2_balanceado", "Milho: 3 regimes irrestritos bal 95 mun", "bal95")

cat("\n=== Onda 2.2: Regimes Institucionais IRRESTRITOS (Z x regime + preços centrados no regime) ===\n")
tab(RUL_s_mt, RUL_s_amz, RUL_s_cer, RUL_m_mt, RUL_m_amz, RUL_m_cer,
    headers = c("Soja MT (irrest.)", "Soja Amz", "Soja Cer", "Milho MT (irrest.)", "Milho Amz", "Milho Cer"))

cat("\n=== Onda 2.2b: Painel balanceado 95 mun. (M1/M2 e regimes irrestritos) e regimes + tendência ===\n")
tab(BAL_M1, BAL_M2, RUT_s_mt, RUT_m_mt, BAL_reg_s, BAL_reg_m,
    headers = c("Bal95 M1", "Bal95 M2", "Reg+Tr Soja", "Reg+Tr Milho", "Bal95 Reg S", "Bal95 Reg M"))

# Testes Wald de H0: Z x D_p2 = Z x D_p3 = 0 e diferenças P1 - P2 / P1 - P3
for (item in list(
  list(m = RUL_s_mt,  pref = "xs_r", zpat = "^zs_p[23]$", crop = "soja",  am = "MT"),
  list(m = RUL_s_amz, pref = "xs_r", zpat = "^zs_p[23]$", crop = "soja",  am = "Amazonia"),
  list(m = RUL_s_cer, pref = "xs_r", zpat = "^zs_p[23]$", crop = "soja",  am = "Cerrado"),
  list(m = RUL_m_mt,  pref = "xm_r", zpat = "^zm_p[23]$", crop = "milho", am = "MT"),
  list(m = RUL_m_amz, pref = "xm_r", zpat = "^zm_p[23]$", crop = "milho", am = "Amazonia"),
  list(m = RUL_m_cer, pref = "xm_r", zpat = "^zm_p[23]$", crop = "milho", am = "Cerrado")
)) {
  w <- wald(item$m, keep = item$zpat, print = FALSE)
  add_test("regimes_irrestritos", paste0("Wald Z x regime (", item$am, ")"), item$crop, item$am,
           stat = w$stat, df1 = w$df1, df2 = w$df2, p = w$p)
  b <- coef(item$m); V <- vcov(item$m); df_t <- degrees_freedom(item$m, "t")
  v1 <- paste0(item$pref, "_p1"); v2 <- paste0(item$pref, "_p2"); v3 <- paste0(item$pref, "_p3")
  for (pair in list(list(lbl = "P1 - P2", vb = v2), list(lbl = "P1 - P3", vb = v3))) {
    dif <- b[v1] - b[pair$vb]
    se_d <- sqrt(V[v1, v1] + V[pair$vb, pair$vb] - 2 * V[v1, pair$vb])
    p_d <- 2 * pt(abs(dif / se_d), df = df_t, lower.tail = FALSE)
    add_test("regimes_irrestritos", paste0(pair$lbl, " (", item$am, ")"), item$crop, item$am,
             est = dif, ep = se_d, stat = dif / se_d, df2 = df_t, p = p_d)
  }
}

# Teste formal de diferença Amazônia vs. Cerrado (geral e por regime, na união Amazônia + Cerrado)
d_ac <- d_reg_u %>%
  filter(bioma %in% c(bioma_amz, bioma_cer)) %>%
  mutate(
    is_amz = as.numeric(bioma == bioma_amz),
    xs_amz_d = xs * is_amz, zs_tr_amz = zs_tr * is_amz,
    xm_amz_d = xm * is_amz, zm_tr_amz = zm_tr * is_amz,
    rain_amz = rain * is_amz,
    xs_p1_amz = xs_r_p1 * is_amz, xs_p2_amz = xs_r_p2 * is_amz, xs_p3_amz = xs_r_p3 * is_amz,
    zs_p2_amz = zs_p2 * is_amz,   zs_p3_amz = zs_p3 * is_amz,
    xm_p1_amz = xm_r_p1 * is_amz, xm_p2_amz = xm_r_p2 * is_amz, xm_p3_amz = xm_r_p3 * is_amz,
    zm_p2_amz = zm_p2 * is_amz,   zm_p3_amz = zm_p3 * is_amz
  )

BIO_s <- rec(feols(y_agr ~ xs + xs_amz_d + zs_tr + zs_tr_amz + rain + rain_amz | ibge + bioma^ano, d_ac, vcov = ~ibge),
             "BIO_diff_soja", "onda2_bioma_diff", "Diferenca Amazonia - Cerrado M1 soja", "obs Amz+Cer")
BIO_m <- rec(feols(y_agr ~ xm + xm_amz_d + zm_tr + zm_tr_amz + rain + rain_amz | ibge + bioma^ano, d_ac, vcov = ~ibge),
             "BIO_diff_milho", "onda2_bioma_diff", "Diferenca Amazonia - Cerrado M2 milho", "obs Amz+Cer")
BIO_reg_s <- rec(feols(y_agr ~ xs_r_p1 + xs_r_p2 + xs_r_p3 + zs_p2 + zs_p3 + rain +
                         xs_p1_amz + xs_p2_amz + xs_p3_amz + zs_p2_amz + zs_p3_amz + rain_amz | ibge + bioma^ano,
                       d_ac, vcov = ~ibge),
                 "REG_diff_soja", "onda2_bioma_diff", "Diferenca Amazonia - Cerrado 3 regimes soja", "obs Amz+Cer")
BIO_reg_m <- rec(feols(y_agr ~ xm_r_p1 + xm_r_p2 + xm_r_p3 + zm_p2 + zm_p3 + rain +
                         xm_p1_amz + xm_p2_amz + xm_p3_amz + zm_p2_amz + zm_p3_amz + rain_amz | ibge + bioma^ano,
                       d_ac, vcov = ~ibge),
                 "REG_diff_milho", "onda2_bioma_diff", "Diferenca Amazonia - Cerrado 3 regimes milho", "obs Amz+Cer")

for (bpair in list(list(m = BIO_reg_s, pat = "^xs_p[123]_amz$", crop = "soja"),
                   list(m = BIO_reg_m, pat = "^xm_p[123]_amz$", crop = "milho"))) {
  w <- wald(bpair$m, keep = bpair$pat, print = FALSE)
  add_test("bioma_diff", "Wald conjunto Amz - Cer (3 regimes)", bpair$crop, "Amz+Cer",
           stat = w$stat, df1 = w$df1, df2 = w$df2, p = w$p)
}
cat("\n=== Onda 2.2c: Diferenças formais Amazônia vs. Cerrado ===\n")
tab(BIO_s, BIO_m, BIO_reg_s, BIO_reg_m, headers = c("M1 Amz-Cer", "M2 Amz-Cer", "Reg Soja Amz-Cer", "Reg Milho Amz-Cer"))

# 3) Teste empilhado Agricultura vs. Pastagem (amostra comum N = 2.767 x 2 = 5.534)
d_stack <- bind_rows(
  d_eff %>% mutate(dest = "agr", is_agr = 1, y_dest = y_agr),
  d_eff %>% mutate(dest = "pas", is_agr = 0, y_dest = y_pas)
) %>% mutate(
  xs_agr = xs * is_agr, zs_tr_agr = zs_tr * is_agr,
  xm_agr = xm * is_agr, zm_tr_agr = zm_tr * is_agr,
  rain_agr = rain * is_agr
)

STK_s <- rec(feols(y_dest ~ xs + xs_agr + zs_tr + zs_tr_agr + rain + rain_agr | ibge^dest + bioma^ano^dest,
                   d_stack, vcov = ~ibge),
             "STACK_soja", "onda2_stacked_destinos", "Empilhado Agr vs Past soja", "obs x 2")
STK_m <- rec(feols(y_dest ~ xm + xm_agr + zm_tr + zm_tr_agr + rain + rain_agr | ibge^dest + bioma^ano^dest,
                   d_stack, vcov = ~ibge),
             "STACK_milho", "onda2_stacked_destinos", "Empilhado Agr vs Past milho", "obs x 2")

cat("\n=== Onda 2.3: Diferença formal Agricultura vs. Pastagem (stacked) ===\n")
tab(STK_s, STK_m, headers = c("Soja Agr-Past", "Milho Agr-Past"))

# 4) Pontes em amostra comum
# 4a. PAM 1991-2000 vs. GAEZ yxHr0 na amostra comum efetiva (N = 1.016)
d_pam_comm <- d_camA[fixest::obs(CA_pam), ]
stopifnot(nrow(d_pam_comm) == 1016)
CA_c_gaez <- rec(feols(y_agr ~ xs + zs_tr + rain | ibge + bioma^ano, d_pam_comm, vcov = ~ibge),
                 "CA_comm_gaez_soja", "onda2_pontes", "Soja GAEZ na amostra PAM (N=1016)", "obs PAM>=7a")
CA_c_both <- rec(feols(y_agr ~ xs + xs_pam90 + zs_tr + zs_pam90_tr + rain | ibge + bioma^ano, d_pam_comm, vcov = ~ibge),
                 "CA_comm_both_soja", "onda2_pontes", "Soja GAEZ + PAM na amostra PAM (N=1016)", "obs PAM>=7a")
w_pam <- wald(CA_c_both, keep = "^xs", print = FALSE)
add_test("pontes", "Wald conjunto GAEZ+PAM (N=1016)", "soja", "obs PAM>=7a",
         stat = w_pam$stat, df1 = w_pam$df1, df2 = w_pam$df2, p = w_pam$p)

# 4b. Amostra fiscal (N = 1.135) com especificação alinhada (ibge + bioma^ano + z_tr) e embargos na mesma subamostra
d_fisc_al <- d_camB %>% filter(fisc)
F_ctrl_s <- run("soja", d_fisc_al, "F_base_ctrl_soja", "onda2_pontes", "M1 soja na amostra fiscal (ibge + bioma^ano)", "fiscal")
F_m6_al_s <- rec(feols(y_agr ~ xs + f1 + zs_tr + rain | ibge + bioma^ano, d_fisc_al, vcov = ~ibge),
                 "F_M6_alinhado_soja", "onda2_pontes", "M6 soja alinhado (ibge + bioma^ano + zs_tr)", "fiscal")
F_m7_al_s <- rec(feols(y_agr ~ xs + f1 + f1_ps + f1_zs + f1_xs + zs_tr + rain | ibge + bioma^ano, d_fisc_al, vcov = ~ibge),
                 "F_M7_alinhado_soja", "onda2_pontes", "M7 soja alinhado (ibge + bioma^ano + zs_tr)", "fiscal")
F_emb_sub_s <- rec(feols(y_agr ~ xs + emb_area_l1 + emb_area_xs + zs_tr + rain | ibge + bioma^ano, d_fisc_al, vcov = ~ibge),
                   "F_emb_area_sub_soja", "onda2_pontes", "Embargos area na amostra fiscal (soja)", "fiscal")
F_ctrl_m <- run("milho", d_fisc_al, "F_base_ctrl_milho", "onda2_pontes", "M2 milho na amostra fiscal (ibge + bioma^ano)", "fiscal")
F_m7_al_m <- rec(feols(y_agr ~ xm + f1 + f1_pm + f1_zm + f1_xm + zm_tr + rain | ibge + bioma^ano, d_fisc_al, vcov = ~ibge),
                 "F_M7_alinhado_milho", "onda2_pontes", "M7 milho alinhado (ibge + bioma^ano + zm_tr)", "fiscal")

cat("\n=== Onda 2.4a/4b: Pontes PAM (N=1.016) e Fiscal Alinhado (N=1.135) ===\n")
tab(CA_c_gaez, CA_pam, CA_c_both, F_ctrl_s, F_m6_al_s, F_m7_al_s, F_emb_sub_s,
    headers = c("GAEZ N=1016", "PAM N=1016", "Ambos N=1016", "M1 fisc", "M6 al.", "M7 al.", "Emb. fisc"))

# 4c. Decomposição passo a passo de `ev` (trend->step c/ bioma^ano, depois bioma^ano->ano, depois full ev)
EV_steps <- list()
for (crop in c("soja", "milho")) {
  z <- if (crop == "soja") "zs" else "zm"
  x <- if (crop == "soja") "xs" else "xm"
  for (pcol in c("post05", "post06")) {
    d_ev <- d_obs %>% mutate(p = .data[[pcol]], e_z = .data[[z]] * p)
    s1 <- rec(feols(as.formula(paste("y_agr ~", x, "+ e_z + rain | ibge + bioma^ano")), d_ev, vcov = ~ibge),
              paste0("EV_s1_", pcol, "_", crop), "onda2_ev_decomp", paste(crop, pcol, "passo1 degrau c/ bioma^ano"), "obs")
    s2 <- rec(feols(as.formula(paste("y_agr ~", x, "+ e_z + rain | ibge + ano")), d_ev, vcov = ~ibge),
              paste0("EV_s2_", pcol, "_", crop), "onda2_ev_decomp", paste(crop, pcol, "passo2 FE ano"), "obs")
    EV_steps[[paste(crop, pcol, "s1")]] <- s1
    EV_steps[[paste(crop, pcol, "s2")]] <- s2
  }
}
cat("\n=== Onda 2.4c: Decomposição passo a passo de `ev` ===\n")
tab(EV_steps[["soja post05 s1"]], EV_steps[["soja post05 s2"]], EV_steps[["milho post05 s1"]], EV_steps[["milho post05 s2"]],
    headers = c("Soja p05 s1 (bioma^ano)", "Soja p05 s2 (ano)", "Milho p05 s1 (bioma^ano)", "Milho p05 s2 (ano)"))

# 4d. Ponte CONAB em amostra comum (N = 2.650 efetivos)
d_conab <- d_obs %>% filter(!is.na(x_cst_s))
C_b2650_s <- run("soja",  d_conab, "C_base_2650_soja",  "onda2_pontes", "M1 soja na amostra CONAB (N=2650)",  "obs CONAB")
C_d2650_s <- run("soja",  d_conab, "C_dap_2650_soja",   "onda2_pontes", "soja + DAP na amostra CONAB (N=2650)", "obs CONAB", extra = "x_dap_s")
C_b2650_m <- run("milho", d_conab, "C_base_2650_milho", "onda2_pontes", "M2 milho na amostra CONAB (N=2650)", "obs CONAB")
C_d2650_m <- run("milho", d_conab, "C_dap_2650_milho",  "onda2_pontes", "milho + DAP na amostra CONAB (N=2650)", "obs CONAB", extra = "x_dap_m")
stopifnot(nobs(C_b2650_s) == 2650, nobs(C_b2650_m) == 2650)
cat("\n=== Onda 2.4d: Ponte CONAB (N = 2.650) ===\n")
tab(C_b2650_s, C_d2650_s, C_b2650_m, C_d2650_m, headers = c("Soja M1 2650", "Soja +DAP 2650", "Milho M2 2650", "Milho +DAP 2650"))
cust_df <- bind_rows(RES) %>% filter(grupo %in% c("principal", "custos") | grepl("^C_(base|dap)_2650", id))
write.csv(cust_df, file.path(out_dir, "onda2_custos_dap_conab.csv"), row.names = FALSE, fileEncoding = "UTF-8")

# 5) Embargos: hierarquia completa na Eq. (4) + gradiente marginal e IC95% pelo método delta na Eq. (5)
d_camB2 <- d_camB %>%
  mutate(
    emb_area_ps = lps * emb_area_l1, emb_area_zs = zs * emb_area_l1,
    emb_area_pm = lpm * emb_area_l1, emb_area_zm = zm * emb_area_l1,
    emb_pm = lpm * emb_esp_l1, emb_zm = zm * emb_esp_l1, emb_xm = xm * emb_esp_l1
  )

B2_area_full_s <- rec(feols(y_agr ~ xs + emb_area_l1 + emb_area_ps + emb_area_zs + emb_area_xs + zs_tr + rain | ibge + bioma^ano,
                            d_camB2, vcov = ~ibge),
                      "B2_emb_area_full_soja", "onda2_embargos", "soja + area embargada hierarquia completa", "obs")
B2_area_full_m <- rec(feols(y_agr ~ xm + emb_area_l1 + emb_area_pm + emb_area_zm + emb_area_xm + zm_tr + rain | ibge + bioma^ano,
                            d_camB2, vcov = ~ibge),
                      "B2_emb_area_full_milho", "onda2_embargos", "milho + area embargada hierarquia completa", "obs")
B2_mod_m <- rec(feols(y_agr ~ xm + emb_esp_l1 + emb_pm + emb_zm + emb_xm + zm_tr + rain | ibge + bioma^ano,
                      d_camB2, vcov = ~ibge),
                "B2_emb_mod_milho", "onda2_embargos", "milho + moderacao N embargos hierarquia completa", "obs")

for (ep in list(
  list(m = B2_s1, pat = "^emb_(ps|zs|xs)$", nm = "Wald interacoes N embargos Eq.5", crop = "soja"),
  list(m = B2_mod_m, pat = "^emb_(pm|zm|xm)$", nm = "Wald interacoes N embargos Eq.5", crop = "milho"),
  list(m = B2_area_full_s, pat = "^emb_area_(ps|zs|xs)$", nm = "Wald interacoes Area embargada Eq.4", crop = "soja"),
  list(m = B2_area_full_m, pat = "^emb_area_(pm|zm|xm)$", nm = "Wald interacoes Area embargada Eq.4", crop = "milho")
)) {
  w <- wald(ep$m, keep = ep$pat, print = FALSE)
  add_test("embargos", ep$nm, ep$crop, "obs", stat = w$stat, df1 = w$df1, df2 = w$df2, p = w$p)
}

# Gradiente marginal beta + eta2 * ln(1 + N) na Eq. (5) para soja e milho
d_emb_eff <- d_camB2[idx_m1, ] %>% mutate(n_emb_l1 = round(expm1(emb_esp_l1)))
pts_n <- sort(unique(c(0L, 1L, 2L, 5L, 10L, 20L, 50L, as.integer(quantile(d_emb_eff$n_emb_l1, c(0.5, 0.75, 0.90, 0.95, 1.0))))))

calc_grad <- function(m, bvar, evar, crop_lbl) {
  b <- coef(m); V <- vcov(m); df_t <- degrees_freedom(m, "t"); tcrit <- qt(0.975, df_t)
  map_dfr(pts_n, function(n0) {
    E0 <- log1p(n0)
    g  <- unname(b[bvar] + b[evar] * E0)
    se <- unname(sqrt(V[bvar, bvar] + (E0^2) * V[evar, evar] + 2 * E0 * V[bvar, evar]))
    pval <- 2 * pt(abs(g / se), df = df_t, lower.tail = FALSE)
    tibble(
      cultura = crop_lbl, n_emb_l1 = n0, ln1p_n = E0,
      pct_obs_abaixo = 100 * mean(d_emb_eff$n_emb_l1 <= n0),
      pct_obs_acima  = 100 * mean(d_emb_eff$n_emb_l1 >= n0),
      gradiente = g, ep = se, ic95_inf = g - tcrit * se, ic95_sup = g + tcrit * se, p = pval
    )
  })
}
grad_df <- bind_rows(
  calc_grad(B2_s1, "xs", "emb_xs", "soja"),
  calc_grad(B2_mod_m, "xm", "emb_xm", "milho")
)
b_s1 <- coef(B2_s1)
n_zero_soja <- expm1(-b_s1["xs"] / b_s1["emb_xs"])
cat(sprintf("\n=== Onda 2.5: Embargos — hierarquia Eq. (4) e gradiente marginal Eq. (5) (ponto zero soja N* = %.1f; pct N>=%d = %.1f%%) ===\n",
            n_zero_soja, ceiling(n_zero_soja), 100 * mean(d_emb_eff$n_emb_l1 >= ceiling(n_zero_soja))))
tab(B2_s1, B2_area_full_s, B2_mod_m, B2_area_full_m,
    headers = c("Soja N Emb.", "Soja Area Full", "Milho N Emb.", "Milho Area Full"))
print(as.data.frame(grad_df), row.names = FALSE)
write.csv(grad_df, file.path(out_dir, "onda2_embargos_gradiente.csv"), row.names = FALSE, fileEncoding = "UTF-8")

# 6) Projeção FWL exata com HAC na série de scores anuais + Driscoll-Kraay + Não-linearidade por tercis de Z
fwl_hac <- function(crop) {
  xvar <- if (crop == "soja") "xs" else "xm"
  ztr  <- if (crop == "soja") "zs_tr" else "zm_tr"
  m_ref <- if (crop == "soja") M1 else M2
  ry <- resid(feols(as.formula(paste("y_agr ~", ztr, "+ rain | ibge + bioma^ano")), data = d_eff))
  rx <- resid(feols(as.formula(paste(xvar, "~", ztr, "+ rain | ibge + bioma^ano")), data = d_eff))
  beta_fwl <- sum(rx * ry) / sum(rx^2)
  stopifnot(abs(beta_fwl - coef(m_ref)[xvar]) < 1e-10)
  e_fwl <- ry - beta_fwl * rx
  yr_df <- tibble(ano = d_eff$ano, prod_xy = rx * ry, denom = rx^2, score = rx * e_fwl) %>%
    group_by(ano) %>%
    summarise(S_t = sum(prod_xy), D_t = sum(denom), g_t = sum(score), .groups = "drop") %>%
    arrange(ano)
  T_yr <- nrow(yr_df)
  m_sc <- lm(g_t ~ 1, data = yr_df)
  V_nw2 <- NeweyWest(m_sc, lag = 2, prewhite = FALSE)
  se_nw2 <- sqrt(as.numeric(V_nw2) * (T_yr^2) / (sum(yr_df$D_t)^2))
  p_nw2  <- 2 * pt(abs(beta_fwl / se_nw2), df = T_yr - 1, lower.tail = FALSE)
  RES[[length(RES) + 1]] <<- tibble(
    id = paste0("R_fwl_hac_", crop), grupo = "onda2_fwl_hac",
    descricao = paste(crop, "FWL exato 24 scores (Newey-West lag 2)"), amostra = "obs",
    termo = xvar, estimativa = beta_fwl, erro_padrao = se_nw2, p = p_nw2, n = nrow(d_eff),
    r2_total = as.numeric(r2(m_ref, "r2")), r2_within = as.numeric(r2(m_ref, "wr2"))
  )
  m_dk2 <- summary(m_ref, vcov = DK(2) ~ ano)
  rec(m_dk2, paste0("R_dk2_", crop), "onda2_fwl_hac", paste(crop, "Driscoll-Kraay lag 2"), "obs")
  cat(sprintf("FWL-HAC (%-5s): beta = %.4f | EP score-HAC(2) = %.4f (p = %.4f) | EP DK(2) = %.4f (p = %.4f)\n",
              crop, beta_fwl, se_nw2, p_nw2, coeftable(m_dk2)[xvar, 2], coeftable(m_dk2)[xvar, 4]))
}
cat("\n=== Onda 2.6: Projeção FWL exata com HAC nos 24 scores e Driscoll-Kraay ===\n")
fwl_hac("soja"); fwl_hac("milho")

# Não-linearidade em Z por tercis municipais (sobre os 141 municípios)
mun_terc <- mun_v2 %>% mutate(terc_s = ntile(apt_s, 3), terc_m = ntile(apt_m, 3))
d_terc <- d_obs %>%
  left_join(mun_terc %>% select(ibge_int, terc_s, terc_m), by = "ibge_int") %>%
  mutate(
    s_t2 = as.numeric(terc_s == 2), s_t3 = as.numeric(terc_s == 3),
    m_t2 = as.numeric(terc_m == 2), m_t3 = as.numeric(terc_m == 3),
    lps_t2 = lps * s_t2, lps_t3 = lps * s_t3, tr_s_t2 = trend * s_t2, tr_s_t3 = trend * s_t3,
    lpm_t2 = lpm * m_t2, lpm_t3 = lpm * m_t3, tr_m_t2 = trend * m_t2, tr_m_t3 = trend * m_t3
  )
NL_s <- rec(feols(y_agr ~ lps_t2 + lps_t3 + tr_s_t2 + tr_s_t3 + rain | ibge + bioma^ano, d_terc, vcov = ~ibge),
            "NL_terc_soja", "onda2_nao_linearidade", "Soja: tercis de aptidao GAEZ (ref = T1)", "obs")
NL_m <- rec(feols(y_agr ~ lpm_t2 + lpm_t3 + tr_m_t2 + tr_m_t3 + rain | ibge + bioma^ano, d_terc, vcov = ~ibge),
            "NL_terc_milho", "onda2_nao_linearidade", "Milho: tercis de aptidao GAEZ (ref = T1)", "obs")

for (nl in list(list(m = NL_s, v2 = "lps_t2", v3 = "lps_t3", pat = "^lps_t[23]$", crop = "soja"),
                list(m = NL_m, v2 = "lpm_t2", v3 = "lpm_t3", pat = "^lpm_t[23]$", crop = "milho"))) {
  w <- wald(nl$m, keep = nl$pat, print = FALSE)
  add_test("nao_linearidade", "Wald conjunto T2=T3=0 (ref T1)", nl$crop, "obs",
           stat = w$stat, df1 = w$df1, df2 = w$df2, p = w$p)
  b <- coef(nl$m); V <- vcov(nl$m); df_t <- degrees_freedom(nl$m, "t")
  d32 <- b[nl$v3] - b[nl$v2]; se32 <- sqrt(V[nl$v3, nl$v3] + V[nl$v2, nl$v2] - 2 * V[nl$v3, nl$v2])
  p32 <- 2 * pt(abs(d32 / se32), df = df_t, lower.tail = FALSE)
  add_test("nao_linearidade", "Contraste T3 - T2", nl$crop, "obs",
           est = d32, ep = se32, stat = d32 / se32, df2 = df_t, p = p32)
}
cat("\n=== Onda 2.6b: Não-linearidade por tercis de aptidão GAEZ (ref = Tercil 1) ===\n")
tab(NL_s, NL_m, headers = c("Soja Tercis Z", "Milho Tercis Z"))

# Consolidar testes conjuntos e exportar CSV auxiliar de custos
tests_df <- bind_rows(TESTS_O2)
write.csv(tests_df, file.path(out_dir, "onda2_testes_conjuntos.csv"), row.names = FALSE, fileEncoding = "UTF-8")
cat("\n=== Resumo de Testes Conjuntos e Contrastes (Onda 2) ===\n")
print(as.data.frame(tests_df), row.names = FALSE)


=== INICIANDO SEÇÃO 2.8b (ONDA 2) ===



=== Onda 2.1: Amostra efetiva exata (N = 2.767) por regime ===


      regime    n n_mun media_y_agr  dp_y_agr mediana_y_agr media_y_pas media_nativa_ha razao_somas_pp     pct_n
  0_total_M1 2767   134  0.11240928 0.3753844    0.01424024   1.1921398        362237.7     0.08231903 100.00000
 1_2001-2005  578   125  0.30062300 0.7529314    0.02597154   2.0492963        393710.8     0.21938272  20.88905
 2_2006-2011  667   119  0.06145369 0.1454203    0.01068653   0.9798364        346862.1     0.04106950  24.10553
 3_2012-2024 1522   134  0.06326333 0.1221511    0.01423022   0.9596627        357023.6     0.04248115  55.00542



=== Onda 2.1: Dispersão dos log-preços (24 anos vs. por regime) ===


         regime n_anos  dp_ln_ps amp_ln_ps   dp_ln_pm amp_ln_pm
 0_total_24anos     24 0.2445618 0.8517496 0.28926760 0.9445918
    1_2001-2005      5 0.1562917 0.3846294 0.06757483 0.1568266
    2_2006-2011      6 0.2301188 0.5969093 0.25491010 0.7167946
    3_2012-2024     13 0.1971501 0.5853052 0.27363464 0.7230757



=== Onda 2.1b: Balanço incluídas (2.767) vs. excluídas (617) ===


... (22 outputs left out)


=== Onda 2.6b: Não-linearidade por tercis de aptidão GAEZ (ref = Tercil 1) ===


                    Soja Tercis Z      Milho Tercis Z
lps_t2           0.301*** (0.104)                    
lps_t3           0.302*** (0.105)                    
tr_s_t2         0.144*** (0.0396)                    
tr_s_t3         0.137*** (0.0362)                    
rain          -0.0133** (0.00571) -0.0152** (0.00639)
lpm_t2                                0.0396 (0.0575)
lpm_t3                              0.112*** (0.0409)
tr_m_t2                               0.0156 (0.0500)
tr_m_t3                             0.113*** (0.0329)
N                            2767                2767
R2 total                   0.4021              0.3844
R2 within                  0.0504              0.0222
Efeitos fixos    ibge + bioma^ano    ibge + bioma^ano
Estimativa (erro-padrão agrupado por município, salvo indicação); * p<0,10  ** p<0,05  *** p<0,01



=== Resumo de Testes Conjuntos e Contrastes (Onda 2) ===


               grupo                               teste cultura     amostra   estimativa erro_padrao estatistica df1 df2           p nota
 regimes_irrestritos                Wald Z x regime (MT)    soja          MT           NA          NA  4.11018329   2 133 0.018533668 <NA>
 regimes_irrestritos                        P1 - P2 (MT)    soja          MT  0.041671232  0.11841090  0.35192057  NA 133 0.725454803 <NA>
 regimes_irrestritos                        P1 - P3 (MT)    soja          MT  0.056780417  0.11428864  0.49681594  NA 133 0.620139066 <NA>
 regimes_irrestritos          Wald Z x regime (Amazonia)    soja    Amazonia           NA          NA  1.34656409   2  70 0.266786510 <NA>
 regimes_irrestritos                  P1 - P2 (Amazonia)    soja    Amazonia  0.214201422  0.16764770  1.27768779  NA  70 0.205580677 <NA>
 regimes_irrestritos                  P1 - P3 (Amazonia)    soja    Amazonia  0.140801488  0.14075405  1.00033702  NA  70 0.320592947 <NA>
 regimes_irrestritos       

***
##### 2.9 Consolidação Geral, Exportação para Auditoria e Veredito (Etapa 15)

Exporta todos os modelos estimados nas Seções 2.2 a 2.8 para `2_INTI/audit/modelos_h1_prec_v2_20261110/resultados_modelos.csv` e avalia os 6 critérios pré-especificados em `PRE_ESPECIFICACAO.md`.

In [12]:
res_all <- bind_rows(RES)
write.csv(res_all, file.path(out_dir, "resultados_modelos.csv"), row.names = FALSE, fileEncoding = "UTF-8")
cat("Total exportado para resultados_modelos.csv:", n_distinct(res_all$id), "modelos/testes;", nrow(res_all), "linhas.\n")

get1 <- function(id, termo) res_all %>% filter(id == !!id, termo == !!termo) %>% slice(1)
veredito <- function(crop) {
  x <- if (crop == "soja") "xs" else "xm"; xf <- paste0(x, "_f")
  pri <- get1(ifelse(crop == "soja", "M1", "M2"), x)
  cl2 <- get1(paste0("R_cl2_", crop), x)
  tws <- get1(paste0("R_2step_", crop), "lp")
  lo  <- res_all %>% filter(grepl(paste0("^R_loyo_", crop, "_"), id), termo == x)
  pl  <- get1(paste0("R_plac_", crop), xf)
  w06 <- get1(paste0("R_w06_", crop), x)
  crit <- tibble(
    criterio = c("C0 principal: estimativa > 0 e p < 0,10",
                 "C1 cluster município+ano: mesmo sinal e p < 0,10",
                 "C2 regressão em 24 pontos: mesmo sinal e p < 0,10",
                 "C3 exclusão de um ano: mesmo sinal em 24 e p < 0,10 em >= 19",
                 "C4 placebo: preço futuro com p >= 0,10",
                 "C5 2006-2024: mesmo sinal"),
    atendido = c(pri$estimativa > 0 & pri$p < 0.10,
                 sign(cl2$estimativa) == sign(pri$estimativa) & cl2$p < 0.10,
                 sign(tws$estimativa) == sign(pri$estimativa) & tws$p < 0.10,
                 all(sign(lo$estimativa) == sign(pri$estimativa)) & sum(lo$p < 0.10) >= 19,
                 pl$p >= 0.10,
                 sign(w06$estimativa) == sign(pri$estimativa)),
    valor = c(sprintf("est %.4f p %.3f", pri$estimativa, pri$p),
              sprintf("est %.4f p %.3f", cl2$estimativa, cl2$p),
              sprintf("est %.4f p %.3f", tws$estimativa, tws$p),
              sprintf("sinais iguais %d/24; p<0,10 em %d", sum(sign(lo$estimativa) == sign(pri$estimativa)), sum(lo$p < 0.10)),
              sprintf("est %.4f p %.3f", pl$estimativa, pl$p),
              sprintf("est %.4f p %.3f", w06$estimativa, w06$p))
  )
  classe <- if (!crit$atendido[1]) "Não consistente" else if (all(crit$atendido)) "Consistente" else "Parcial"
  list(crit = crit, classe = classe)
}
vs <- veredito("soja"); vm <- veredito("milho")
cat("\n=== VEREDITO PRÉ-ESPECIFICADO — SOJA (M1):", vs$classe, "===\n"); print(as.data.frame(vs$crit), row.names = FALSE)
cat("\n=== VEREDITO PRÉ-ESPECIFICADO — MILHO (M2):", vm$classe, "===\n"); print(as.data.frame(vm$crit), row.names = FALSE)
write.csv(bind_rows(mutate(vs$crit, cultura = "soja", classe = vs$classe), mutate(vm$crit, cultura = "milho", classe = vm$classe)),
          file.path(out_dir, "veredito_criterio.csv"), row.names = FALSE, fileEncoding = "UTF-8")

Total exportado para resultados_modelos.csv: 178 modelos/testes; 662 linhas.



=== VEREDITO PRÉ-ESPECIFICADO — SOJA (M1): Parcial ===


                                                     criterio atendido                             valor
                      C0 principal: estimativa > 0 e p < 0,10     TRUE                est 0.0848 p 0.019
             C1 cluster município+ano: mesmo sinal e p < 0,10     TRUE                est 0.0848 p 0.091
            C2 regressão em 24 pontos: mesmo sinal e p < 0,10     TRUE                est 0.1344 p 0.052
 C3 exclusão de um ano: mesmo sinal em 24 e p < 0,10 em >= 19     TRUE sinais iguais 24/24; p<0,10 em 24
                       C4 placebo: preço futuro com p >= 0,10    FALSE                est 0.0844 p 0.014
                                    C5 2006-2024: mesmo sinal    FALSE               est -0.0067 p 0.419



=== VEREDITO PRÉ-ESPECIFICADO — MILHO (M2): Parcial ===


                                                     criterio atendido                             valor
                      C0 principal: estimativa > 0 e p < 0,10     TRUE                est 0.0519 p 0.004
             C1 cluster município+ano: mesmo sinal e p < 0,10     TRUE                est 0.0519 p 0.074
            C2 regressão em 24 pontos: mesmo sinal e p < 0,10     TRUE                est 0.0967 p 0.069
 C3 exclusão de um ano: mesmo sinal em 24 e p < 0,10 em >= 19     TRUE sinais iguais 24/24; p<0,10 em 24
                       C4 placebo: preço futuro com p >= 0,10    FALSE                est 0.1026 p 0.001
                                    C5 2006-2024: mesmo sinal    FALSE               est -0.0040 p 0.613
